# Facilities Silver: Six-Source Facility Merge

Six-way merge of all facility bronze tables: NHFR (1,359), infirmary (614), hospital list (1,336), primary care (1,180), dialysis clinics (800), and blood service (121) from `public_health.bronze`. Joined on `UPPER(TRIM(facility_name))` with ROW_NUMBER dedup per source. Unmatched records from any source are included with source flags (e.g. `PRIMARY_CARE`, `DIALYSIS`, `BLOOD_SERVICE`). Overlapping fields across all sources are verified with traceable comparison columns.

In [0]:
-- =============================================================================
-- facilities_silver_01_registry.sql
-- Builds two tables, one build id each:
--   public_health.silver.facilities_id_map_silver    one row per source identifier
--                                                     key (_source_key, source_record_id)
--   public_health.silver.facilities_registry_silver  one row per facility establishment
--                                                     key facility_key
-- Follows facilities-silver-plan.md (sections 2, 4, 5, 6) and lineage-qc-contract.md v1.2.
--
-- Inputs: NHFR hospitals (spine) and four HFSRB licensed lists (hospitals, infirmaries,
-- primary care, free-standing dialysis clinics), geography silver, the two reference tables
-- from facilities_silver_00_reference.sql, and the PREVIOUS versions of the two targets
-- (so facility_key and history columns survive a rebuild).
--
-- Build sequence (contract section 5.1)
--   Step 1, inputs
--     [0]  build ids, settings, helper functions, geography reference view
--     [1]  ..__classified   every bronze row of the five sources, parsed, with row_class
--     [2]  ..__inputs       one row per file or sheet read, with load id and checksum
--   Step 2, transform
--     [3]  ..__geo          area of every data row: from codes (NHFR) or name + level + parent
--     [4]  ..__records      identity: duplicate-code rule, code match, name match, overrides,
--                           facility_key (reused from the previous id map, else a new uuid)
--     [5]  ..__idmap_candidate, ..__candidate   the two candidate tables
--   Step 3, validate
--     [6]  ..__gates        every row must have failures = 0
--     [7]  build log rows   (succeeded or failed)
--     [8]  STOP if a gate failed: the failed build is logged, nothing else is written
--   Step 4, record
--     [9]  write both tables from the validated candidates
--     [10] append build_log, build_inputs, row_disposition (rows not loaded only, contract v1.3)
--     [11] summary; drop staging tables
--
-- Write mode: the candidate is the full new state of each table (current rows + rows carried
-- from the previous version), validated, then written with CREATE OR REPLACE. This gives the
-- same result as the MERGE described in the plan (section 5.3), and the rows validated are
-- exactly the rows written. Delta keeps every version.
--
-- Needs
--   facilities_silver_00_reference.sql has been run
--   quality_setup.sql has been run (build_log, build_inputs, row_disposition)
--   geography silver exists (name set in [0], default public_health.silver.geography_silver)
--   Bronze QC passed for the five sources
--
-- Status
--   Ran on local Spark 4.0 against the landing files and the exported geography_silver
--   (2026-10-09), twice in a row (first build, then a rebuild that reuses keys).
--   NOT tested locally: Delta, read_files(), and DESCRIBE HISTORY in a subquery ([2], [10]).
--   If your runtime rejects DESCRIBE HISTORY there, use the NULL version noted in [2].
-- =============================================================================

-- [0] build ids and settings
DECLARE OR REPLACE VARIABLE registry_build_id STRING DEFAULT uuid();
DECLARE OR REPLACE VARIABLE idmap_build_id STRING DEFAULT uuid();
DECLARE OR REPLACE VARIABLE silver_build_started_at TIMESTAMP DEFAULT current_timestamp();
-- true: every input needs exactly one processed ingestion_metadata row (contract 3.5)
DECLARE OR REPLACE VARIABLE strict_load_lookup BOOLEAN DEFAULT true;
DECLARE OR REPLACE VARIABLE code_version STRING DEFAULT NULL;   -- Git commit, if you pass one
DECLARE OR REPLACE VARIABLE job_run_id STRING DEFAULT NULL;     -- {{job.run_id}} from a job
-- geography snapshot: the latest one in geography silver (psgc_code is unique per snapshot)
DECLARE OR REPLACE VARIABLE geography_checksum STRING;
SET VARIABLE geography_checksum =
  (SELECT max_by(content_checksum, silver_processed_at) FROM public_health.silver.geography_silver);

-- Text normalisation: upper case; N-tilde (also the broken 'Ã±' and NHFR's '?') becomes N;
-- anything that is not A-Z or 0-9 becomes a space; single spaces.
CREATE OR REPLACE TEMPORARY FUNCTION fac_norm(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(
         translate(upper(replace(replace(replace(replace(s, 'Ã±', 'N'), 'Ã‘', 'N'), 'ñ', 'N'), '?', 'N')), 'ÑÁÉÍÓÚ', 'NAEIOU'),
         '[^A-Z0-9]', ' '), ' +', ' '));

-- Place name for matching: 'CITY OF LAS PIÑAS (HUC)' and 'Las Piñas City' -> 'LAS PINAS';
-- 'Sto. Tomas' and 'Santo Tomas' -> 'SANTO TOMAS'; 'Municipality of Glan' -> 'GLAN';
-- 'Island Garden City of Samal' and 'Samal City' -> 'SAMAL'. Applied to both sides.
-- Apostrophes and hyphens are dropped first ('M'Lang' = 'Mlang', 'Jala-Jala' = 'Jalajala'), and
-- fac_place_key compares without spaces ('Mataas na Kahoy' = 'Mataasnakahoy').
CREATE OR REPLACE TEMPORARY FUNCTION fac_place(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
         regexp_replace(regexp_replace(concat(' ', fac_norm(regexp_replace(s, "['’`-]", '')), ' '),
         ' (HUC|ICC|CC|CAPITAL|NOT A PROVINCE) ', ' '),
         ' STA ', ' SANTA '), ' STO ', ' SANTO '), ' GEN ', ' GENERAL '), ' GOV ', ' GOVERNOR '), ' SEN ', ' SENATOR '),
         '^ (MUNICIPALITY OF|ISLAND GARDEN CITY OF|SCIENCE CITY OF|CITY OF) ', ' '), ' CITY $', ' '), ' +', ' '));

CREATE OR REPLACE TEMPORARY FUNCTION fac_place_key(s STRING) RETURNS STRING
RETURN replace(fac_place(s), ' ', '');

-- True when a label says it is a city
CREATE OR REPLACE TEMPORARY FUNCTION fac_city_hint(s STRING) RETURNS BOOLEAN
RETURN fac_norm(s) RLIKE '(^CITY OF )|( CITY$)|( CITY )';

-- Facility name for matching: normalised, common abbreviations expanded, legal suffixes dropped
CREATE OR REPLACE TEMPORARY FUNCTION fac_fname(s STRING) RETURNS STRING
RETURN trim(regexp_replace(
  regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
    concat(' ', fac_norm(s), ' '),
    ' HOSP ', ' HOSPITAL '), ' MED ', ' MEDICAL '), ' (CTR|CNTR) ', ' CENTER '), ' GEN ', ' GENERAL '),
    ' MEM ', ' MEMORIAL '), ' STO ', ' SANTO '), ' (INC|CORP|CORPORATION|CO) ', ' '),
  ' +', ' '));

-- Distinctive words of a facility name (generic words removed), for comparing two names
CREATE OR REPLACE TEMPORARY FUNCTION fac_name_tokens(s STRING) RETURNS ARRAY<STRING>
RETURN array_distinct(filter(split(fac_fname(s), ' '), t -> length(t) > 1 AND NOT array_contains(array(
  'HOSPITAL', 'MEDICAL', 'CENTER', 'GENERAL', 'DISTRICT', 'PROVINCIAL', 'MEMORIAL', 'REGIONAL', 'CITY', 'OF', 'THE',
  'AND', 'DE', 'DEL', 'NG', 'SA', 'MATERNITY', 'CLINIC', 'INFIRMARY', 'HEALTH', 'CARE', 'DOCTORS', 'DR', 'ST',
  'SANTO', 'SAN', 'OPC', 'COMMUNITY', 'MUNICIPAL', 'HOSPITALS', 'SPECIALTY', 'FOUNDATION'), t)));

-- Share of distinctive words two names have in common (0 to 1; null if either has none)
CREATE OR REPLACE TEMPORARY FUNCTION fac_name_similarity(a STRING, b STRING) RETURNS DOUBLE
RETURN CASE WHEN size(fac_name_tokens(a)) = 0 OR size(fac_name_tokens(b)) = 0 THEN NULL
            ELSE size(array_intersect(fac_name_tokens(a), fac_name_tokens(b)))
                 / least(size(fac_name_tokens(a)), size(fac_name_tokens(b))) END;

-- LGU of a city / municipality / sub-municipality code: a Manila district (1380601000 Tondo)
-- belongs to its city (1380600000). Codes of other levels are returned unchanged.
CREATE OR REPLACE TEMPORARY FUNCTION fac_lgu(code STRING) RETURNS STRING
RETURN CASE WHEN code RLIKE '^13806[0-9]{2}000$' THEN '1380600000' ELSE code END;

-- Text to number: spaces and thousands separators removed; null when not a number
CREATE OR REPLACE TEMPORARY FUNCTION fac_num(s STRING) RETURNS DECIMAL(20,4)
RETURN try_cast(regexp_replace(trim(s), '[ ,]', '') AS DECIMAL(20,4));

-- NHFR id from any of the formats seen: 3880, 03880, 3880.0, DOH000000000003880, DOH03880.
-- Compared as a number, never as a padded string (plan 2.2). Null for anything else.
CREATE OR REPLACE TEMPORARY FUNCTION fac_nhfr_id(s STRING) RETURNS BIGINT
RETURN CASE WHEN trim(s) RLIKE '^(DOH|doh)?[0-9]{1,15}([.]0)?$'
            THEN try_cast(regexp_extract(trim(s), '([0-9]{1,15})', 1) AS BIGINT) END;

-- Date from ISO text (2026-12-31T00:00:00) or MM/dd/yyyy
CREATE OR REPLACE TEMPORARY FUNCTION fac_date(s STRING) RETURNS DATE
RETURN coalesce(to_date(try_to_timestamp(trim(s))), to_date(try_to_timestamp(trim(s), 'MM/dd/yyyy')),
                to_date(try_to_timestamp(trim(s), 'M/d/yyyy')));

-- Region code from any region label seen in the facilities sources (NCR, CORDILLERA, NIR,
-- REGION 4-A, Region IV-A  CALABARZON, REGION XII (SOCCSKSARGEN), ARMM, N C R, ...).
-- Null when the label is not recognised (fails a gate). ARMM maps to BARMM (19).
CREATE OR REPLACE TEMPORARY FUNCTION fac_region_psgc(s STRING) RETURNS STRING
RETURN CASE
  WHEN fac_norm(s) RLIKE '(^| )(NCR|N C R|NATIONAL CAPITAL|METRO MANILA)( |$)'           THEN '1300000000'
  WHEN fac_norm(s) RLIKE '(^| )(CAR|C A R|CORDILLERA)( |$)'                               THEN '1400000000'
  WHEN fac_norm(s) RLIKE '(^| )(BARMM|ARMM|BANGSAMORO|AUTONOMOUS REGION IN MUSLIM MINDANAO)( |$)' THEN '1900000000'
  WHEN fac_norm(s) RLIKE '(^| )(NIR|NEGROS ISLAND)( |$)'                                  THEN '1800000000'
  WHEN fac_norm(s) RLIKE '(^| )(MIMAROPA|IV B|4 B|4B|IVB)( |$)'                           THEN '1700000000'
  WHEN fac_norm(s) RLIKE '(^| )(CARAGA|XIII|13)( |$)'                                     THEN '1600000000'
  WHEN fac_norm(s) RLIKE '(^| )(CALABARZON|IV A|4 A|4A|IVA)( |$)'                         THEN '0400000000'
  -- region names without a number (FHSIS 2019-2020: ILOCOS, CAGAYAN VALLEY, ...)
  WHEN fac_norm(s) RLIKE '^ILOCOS( REGION)?$'          THEN '0100000000'
  WHEN fac_norm(s) RLIKE '^CAGAYAN VALLEY( REGION)?$'  THEN '0200000000'
  WHEN fac_norm(s) RLIKE '^CENTRAL LUZON( REGION)?$'   THEN '0300000000'
  WHEN fac_norm(s) RLIKE '^BICOL( REGION)?$'           THEN '0500000000'
  WHEN fac_norm(s) RLIKE '^WESTERN VISAYAS( REGION)?$' THEN '0600000000'
  WHEN fac_norm(s) RLIKE '^CENTRAL VISAYAS( REGION)?$' THEN '0700000000'
  WHEN fac_norm(s) RLIKE '^EASTERN VISAYAS( REGION)?$' THEN '0800000000'
  WHEN fac_norm(s) RLIKE '^ZAMBOANGA PENINSULA( REGION)?$' THEN '0900000000'
  WHEN fac_norm(s) RLIKE '^NORTHERN MINDANAO( REGION)?$'   THEN '1000000000'
  WHEN fac_norm(s) RLIKE '^DAVAO( REGION)?$'           THEN '1100000000'
  WHEN fac_norm(s) RLIKE '^SOCCSKSARGEN( REGION)?$'    THEN '1200000000'
  ELSE CASE regexp_extract(fac_norm(s), '^REGION ([IVX]+|[0-9]+)( |$)', 1)
         WHEN 'I' THEN '0100000000' WHEN '1' THEN '0100000000' WHEN 'II' THEN '0200000000' WHEN '2' THEN '0200000000'
         WHEN 'III' THEN '0300000000' WHEN '3' THEN '0300000000' WHEN 'V' THEN '0500000000' WHEN '5' THEN '0500000000'
         WHEN 'VI' THEN '0600000000' WHEN '6' THEN '0600000000' WHEN 'VII' THEN '0700000000' WHEN '7' THEN '0700000000'
         WHEN 'VIII' THEN '0800000000' WHEN '8' THEN '0800000000' WHEN 'IX' THEN '0900000000' WHEN '9' THEN '0900000000'
         WHEN 'X' THEN '1000000000' WHEN '10' THEN '1000000000' WHEN 'XI' THEN '1100000000' WHEN '11' THEN '1100000000'
         WHEN 'XII' THEN '1200000000' WHEN '12' THEN '1200000000' END
  END;

-- Geography reference: one snapshot; level names, place name for matching, parents.
-- province_psgc is null for highly urbanized cities (they have no province).
CREATE OR REPLACE TEMP VIEW fac_geo AS
SELECT psgc_code, area_name,
       CASE geographic_level WHEN 'Reg' THEN 'region' WHEN 'Prov' THEN 'province' WHEN 'City' THEN 'city'
            WHEN 'Mun' THEN 'municipality' WHEN 'SubMun' THEN 'submunicipality' WHEN 'Bgy' THEN 'barangay' END AS geo_level,
       city_class,
       fac_place_key(area_name) AS place_norm,
       concat(substr(psgc_code, 1, 2), '00000000') AS region_psgc,
       CASE WHEN geographic_level IN ('City', 'Mun') AND coalesce(city_class, '') <> 'HUC'
            THEN concat(substr(psgc_code, 1, 5), '00000') END AS province_psgc
FROM public_health.silver.geography_silver
WHERE content_checksum = geography_checksum;

-- Target tables must exist before the first build (the build reads their previous version).
CREATE TABLE IF NOT EXISTS public_health.silver.facilities_id_map_silver (
  _source_key STRING, source_record_id STRING, source_id_raw STRING, source_id_norm BIGINT,
  code_validated BOOLEAN, facility_key STRING, match_method STRING, match_status STRING,
  _first_build_id STRING, _silver_build_id STRING, _source_file STRING, _source_sheet STRING,
  _source_row_num INT, _as_of_date DATE);

CREATE TABLE IF NOT EXISTS public_health.silver.facilities_registry_silver (
  facility_key STRING, facility_category STRING, facility_name STRING, nhfr_id BIGINT, license_no STRING,
  ownership STRING, ownership_class STRING, service_level STRING,
  area_key STRING, psgc_code STRING, barangay_psgc STRING, geo_name_source STRING, geo_resolution_method STRING,
  geo_conflict_nhfr_hfsrb BOOLEAN,
  nhfr_reported_beds INT, hfsrb_abc INT, authorized_beds INT, analysis_beds INT, analysis_beds_basis STRING,
  dialysis_stations INT, has_blood_service BOOLEAN, blood_service_type STRING,
  nhfr_licensing_status STRING, license_valid_until DATE, is_license_valid_on_as_of BOOLEAN,
  in_nhfr BOOLEAN, in_hfsrb BOOLEAN, match_method STRING,
  is_currently_listed_nhfr BOOLEAN, is_currently_listed_hfsrb BOOLEAN, facility_status STRING,
  _row_hash STRING, _first_build_id STRING, _business_change_build_id STRING, _delisted_build_id STRING,
  _silver_build_id STRING,
  _nhfr_source_key STRING, _nhfr_source_file STRING, _nhfr_source_row_num INT, _nhfr_as_of_date DATE,
  _hfsrb_source_key STRING, _hfsrb_source_file STRING, _hfsrb_source_sheet STRING, _hfsrb_source_row_num INT,
  _hfsrb_as_of_date DATE,
  _metric_source_columns MAP<STRING, STRING>);


-- =============================================================================
-- [1] classify and parse every bronze row of the five sources
--     row_class: data / non_data. Identity decisions (duplicate, quarantined) come in [4].
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_nhfr_rows AS
SELECT 'public_health.bronze.facilities_nhfr_bronze' AS input_table,
       _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       'hospital' AS facility_category,
       Health_Facility_Code AS source_id_raw,
       CASE WHEN trim(Health_Facility_Code) RLIKE '^DOH[0-9]{15}$'
            THEN CAST(substr(trim(Health_Facility_Code), 4) AS BIGINT) END AS nhfr_id,
       trim(Facility_Name) AS facility_name,
       Region_Name AS region_label, Province_Name AS province_label, `City/Municipality_Name` AS city_label,
       `City/Municipality_PSGC` AS city_code_raw, Barangay_PSGC AS barangay_code_raw,
       initcap(trim(Ownership_Major_Classification)) AS ownership,
       trim(coalesce(`Ownership_Sub-Classification_for_Government_facilities`,
                     `Ownership_Sub-Classification_for_private_facilities`)) AS ownership_class,
       trim(Service_Capability) AS service_level,
       CAST(fac_num(Bed_Capacity) AS INT) AS nhfr_reported_beds,
       CAST(NULL AS INT) AS hfsrb_abc, CAST(NULL AS INT) AS dialysis_stations, CAST(NULL AS STRING) AS blood_service_type,
       trim(regexp_replace(Licensing_Status, '<[^>]*>|Licensing Status:', '')) AS nhfr_licensing_status,
       CAST(NULL AS DATE) AS license_valid_until, CAST(NULL AS STRING) AS license_no,
       map('facility_name', 'Facility_Name', 'nhfr_reported_beds', 'Bed_Capacity', 'service_level', 'Service_Capability',
           'nhfr_licensing_status', 'Licensing_Status', 'ownership', 'Ownership_Major_Classification') AS value_columns,
       CASE WHEN Health_Facility_Code IS NULL AND Facility_Name IS NULL THEN 'non_data' ELSE 'data' END AS row_class,
       CASE WHEN Health_Facility_Code IS NULL AND Facility_Name IS NULL THEN 'blank_row' END AS reason
FROM public_health.bronze.facilities_nhfr_bronze
WHERE _source_key = 'doh_nhfr';

CREATE OR REPLACE TEMP VIEW fac_hfsrb_rows AS
WITH u AS (
  SELECT 'public_health.bronze.facilities_hfsrb_hospitals_bronze' AS input_table, 'hospital' AS facility_category,
         _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
         nhfr_code, facility_name, region, province, city_municipality, ownership, ownership_class,
         service_capability, abc, dialysis_stations AS dialysis_raw, blood_service_facility, validity_until, license_no,
         map('facility_name', 'facility_name', 'hfsrb_abc', 'abc', 'service_level', 'service_capability',
             'dialysis_stations', 'dialysis_stations', 'blood_service_type', 'blood_service_facility',
             'license_valid_until', 'validity_until', 'ownership', 'ownership') AS value_columns
  FROM public_health.bronze.facilities_hfsrb_hospitals_bronze WHERE _source_key = 'doh_hfsrb_hospitals'
  UNION ALL
  SELECT 'public_health.bronze.facilities_hfsrb_infirmaries_bronze', 'infirmary',
         _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
         nhfr_code, facility_name, region, province, city_municipality, ownership, ownership_class,
         CAST(NULL AS STRING), abc, CAST(NULL AS STRING), CAST(NULL AS STRING), validity_until, license_no,
         map('facility_name', 'facility_name', 'hfsrb_abc', 'abc', 'license_valid_until', 'validity_until', 'ownership', 'ownership')
  FROM public_health.bronze.facilities_hfsrb_infirmaries_bronze WHERE _source_key = 'doh_hfsrb_infirmaries'
  UNION ALL
  SELECT 'public_health.bronze.facilities_hfsrb_primary_care_bronze', 'primary_care',
         _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
         nhfr_code, facility_name, region, province, city_municipality, ownership, CAST(NULL AS STRING),
         CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), validity_until, license_no,
         map('facility_name', 'facility_name', 'license_valid_until', 'validity_until', 'ownership', 'ownership')
  FROM public_health.bronze.facilities_hfsrb_primary_care_bronze WHERE _source_key = 'doh_hfsrb_primary_care'
  UNION ALL
  SELECT 'public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze', 'dialysis_clinic',
         _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
         nhfr_code, facility_name, region, province, city_municipality, ownership, CAST(NULL AS STRING),
         CAST(NULL AS STRING), CAST(NULL AS STRING), stations, CAST(NULL AS STRING), validity_until, license_no,
         map('facility_name', 'facility_name', 'dialysis_stations', 'stations', 'license_valid_until', 'validity_until', 'ownership', 'ownership')
  FROM public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze WHERE _source_key = 'doh_hfsrb_dialysis_clinics'
)
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       facility_category,
       nhfr_code AS source_id_raw,
       fac_nhfr_id(nhfr_code) AS nhfr_id,
       trim(facility_name) AS facility_name,
       region AS region_label, province AS province_label, city_municipality AS city_label,
       CAST(NULL AS STRING) AS city_code_raw, CAST(NULL AS STRING) AS barangay_code_raw,
       -- ownership from the column, else from the sheet name (' Private' has a leading space)
       initcap(trim(coalesce(ownership, _source_sheet))) AS ownership,
       trim(ownership_class) AS ownership_class,
       trim(service_capability) AS service_level,
       CAST(NULL AS INT) AS nhfr_reported_beds,
       CAST(fac_num(abc) AS INT) AS hfsrb_abc,
       CAST(fac_num(dialysis_raw) AS INT) AS dialysis_stations,          -- 'N/A', 'none' -> null
       CASE WHEN trim(blood_service_facility) IS NOT NULL THEN initcap(trim(blood_service_facility)) END AS blood_service_type,
       CAST(NULL AS STRING) AS nhfr_licensing_status,
       fac_date(validity_until) AS license_valid_until,
       trim(license_no) AS license_no,
       value_columns,
       CASE WHEN facility_name IS NULL AND nhfr_code IS NULL AND license_no IS NULL THEN 'non_data' ELSE 'data' END AS row_class,
       CASE WHEN facility_name IS NULL AND nhfr_code IS NULL AND license_no IS NULL THEN 'blank_or_footer' END AS reason
FROM u;

CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__classified AS
SELECT *, CASE WHEN source_id_raw IS NOT NULL AND nhfr_id IS NULL THEN 'invalid_code_token' END AS code_issue
FROM fac_nhfr_rows
UNION ALL
SELECT *, CASE WHEN source_id_raw IS NOT NULL AND nhfr_id IS NULL THEN 'invalid_code_token' END
FROM fac_hfsrb_rows;


-- =============================================================================
-- [2] inputs: everything this build reads (layout of quality.build_inputs)
-- =============================================================================
-- Declared inputs (plan 5.4): the build fails if one is missing or extra.
CREATE OR REPLACE TEMP VIEW fac_declared_inputs AS
SELECT * FROM VALUES
  ('doh_nhfr', NULL),
  ('doh_hfsrb_hospitals', 'Government'), ('doh_hfsrb_hospitals', 'Private'),
  ('doh_hfsrb_infirmaries', 'Government'), ('doh_hfsrb_infirmaries', 'Private'),
  ('doh_hfsrb_primary_care', 'Government'), ('doh_hfsrb_primary_care', 'Private'),
  ('doh_hfsrb_dialysis_clinics', 'Government'), ('doh_hfsrb_dialysis_clinics', ' Private')
  AS t(source_key, source_sheet);

-- Delta versions of the tables read. If DESCRIBE HISTORY in a subquery is rejected, replace
-- each (SELECT max(version) ...) with CAST(NULL AS BIGINT).
CREATE OR REPLACE TEMP VIEW fac_registry_table_versions AS
SELECT 'public_health.bronze.facilities_nhfr_bronze' AS table_name,
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_nhfr_bronze)) AS table_version
UNION ALL SELECT 'public_health.bronze.facilities_hfsrb_hospitals_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_hfsrb_hospitals_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_hfsrb_infirmaries_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_hfsrb_infirmaries_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_hfsrb_primary_care_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_hfsrb_primary_care_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_hfsrb_dialysis_clinics_bronze))
UNION ALL SELECT 'public_health.silver.geography_silver',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.geography_silver))
UNION ALL SELECT 'public_health.silver.facilities_area_exceptions',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_area_exceptions))
UNION ALL SELECT 'public_health.silver.facilities_id_map_override',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_id_map_override))
UNION ALL SELECT 'public_health.silver.facilities_id_map_silver',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_id_map_silver))
UNION ALL SELECT 'public_health.silver.facilities_registry_silver',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_registry_silver));

CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__inputs AS
WITH files AS (
  SELECT input_table, _source_key, _source_file, _source_sheet, _ingested_at, count(*) AS input_rows
  FROM public_health.silver.facilities_registry_silver__classified
  GROUP BY input_table, _source_key, _source_file, _source_sheet, _ingested_at
)
SELECT registry_build_id AS build_id,
       'public_health.silver.facilities_registry_silver' AS target_table,
       'bronze_source' AS input_kind,
       f.input_table,
       max(v.table_version) AS input_table_version,
       f._source_key AS source_key, f._source_file AS source_file, f._source_sheet AS source_sheet,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.batch_id) END AS load_batch_id,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.content_checksum) END AS content_checksum,
       f._ingested_at AS ingested_at,
       f.input_rows,
       CAST(NULL AS STRING) AS reference_version,
       count(m.batch_id) AS metadata_matches                         -- gate only, not appended
FROM files f
LEFT JOIN public_health.bronze.ingestion_metadata m
  ON m.source_name = f._source_key
 AND lower(m.status) = 'processed'
 AND endswith(m.raw_file_path, concat('/', f._source_file))
 AND m.source_sheet <=> f._source_sheet
 AND m.retrieved_at = f._ingested_at
LEFT JOIN fac_registry_table_versions v ON v.table_name = f.input_table
GROUP BY f.input_table, f._source_key, f._source_file, f._source_sheet, f._ingested_at, f.input_rows
UNION ALL
-- tables that shaped the build: geography, the two reference tables, the previous targets
SELECT registry_build_id, 'public_health.silver.facilities_registry_silver', 'table', t.table_name, v.table_version,
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CASE WHEN t.table_name LIKE '%geography_silver' THEN geography_checksum END,
       CAST(NULL AS TIMESTAMP), t.n, CAST(NULL AS STRING), CAST(NULL AS BIGINT)
FROM (SELECT 'public_health.silver.geography_silver' AS table_name, count(*) AS n FROM fac_geo
      UNION ALL SELECT 'public_health.silver.facilities_area_exceptions', count(*) FROM public_health.silver.facilities_area_exceptions
      UNION ALL SELECT 'public_health.silver.facilities_id_map_override', count(*) FROM public_health.silver.facilities_id_map_override
      UNION ALL SELECT 'public_health.silver.facilities_id_map_silver', count(*) FROM public_health.silver.facilities_id_map_silver
      UNION ALL SELECT 'public_health.silver.facilities_registry_silver', count(*) FROM public_health.silver.facilities_registry_silver) t
LEFT JOIN fac_registry_table_versions v ON v.table_name = t.table_name;


-- =============================================================================
-- [3] geography: the area of every data row (plan 4.2). Each row resolves once.
--   NHFR: its own city/municipality code (Sulu codes remapped by the exceptions file),
--         validated against geography; the level comes from geography.
--   HFSRB: region label -> region (rule), province label -> province (name within region),
--          city label -> city / municipality / sub-municipality by name within the region,
--          ties broken by the province and by whether the label says "city".
--          An exception row (area_label) wins over the rule.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__geo AS
WITH remap AS (
  SELECT match_value AS from_prefix, psgc_code AS to_prefix
  FROM public_health.silver.facilities_area_exceptions
  WHERE source_group = 'nhfr' AND match_type = 'code_prefix'
),
nhfr AS (
  SELECT c._source_key, c._source_file, c._source_sheet, c._source_row_num,
         coalesce(concat(r1.to_prefix, substr(c.city_code_raw, length(r1.from_prefix) + 1)), c.city_code_raw) AS city_code,
         coalesce(concat(r2.to_prefix, substr(c.barangay_code_raw, length(r2.from_prefix) + 1)), c.barangay_code_raw) AS bgy_code,
         (r1.from_prefix IS NOT NULL) AS remapped
  FROM public_health.silver.facilities_registry_silver__classified c
  LEFT JOIN remap r1 ON startswith(c.city_code_raw, r1.from_prefix)
  LEFT JOIN remap r2 ON startswith(c.barangay_code_raw, r2.from_prefix)
  WHERE c._source_key = 'doh_nhfr' AND c.row_class = 'data'
),
nhfr_geo AS (
  SELECT n._source_key, n._source_file, n._source_sheet, n._source_row_num,
         g.psgc_code, CASE WHEN b.psgc_code IS NOT NULL THEN n.bgy_code END AS barangay_psgc,
         CASE WHEN n.remapped THEN 'exception' ELSE 'source_code' END AS geo_resolution_method,
         CASE WHEN g.psgc_code IS NULL THEN 1 ELSE 0 END AS geo_unresolved,
         CAST(NULL AS STRING) AS geo_candidates
  FROM nhfr n
  LEFT JOIN fac_geo g ON g.psgc_code = n.city_code AND g.geo_level IN ('city', 'municipality', 'submunicipality')
  LEFT JOIN fac_geo b ON b.psgc_code = n.bgy_code AND b.geo_level = 'barangay'
),
h AS (
  SELECT c._source_key, c._source_file, c._source_sheet, c._source_row_num,
         fac_region_psgc(c.region_label) AS region_psgc,
         fac_place_key(c.province_label) AS prov_place, fac_place_key(c.city_label) AS city_place,
         fac_norm(c.city_label) AS city_norm, fac_norm(c.province_label) AS prov_norm,
         fac_city_hint(c.city_label) AS city_hint
  FROM public_health.silver.facilities_registry_silver__classified c
  WHERE c._source_key <> 'doh_nhfr' AND c.row_class = 'data'
),
-- province: exception row, else a unique province of that name in the region, else in the country
prov_region AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num, count(*) AS n, max(p.psgc_code) AS psgc_code
  FROM h JOIN fac_geo p ON p.geo_level = 'province' AND p.region_psgc = h.region_psgc AND p.place_norm = h.prov_place
  GROUP BY h._source_key, h._source_file, h._source_sheet, h._source_row_num
),
prov_any AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num, count(*) AS n, max(p.psgc_code) AS psgc_code
  FROM h JOIN fac_geo p ON p.geo_level = 'province' AND p.place_norm = h.prov_place
  GROUP BY h._source_key, h._source_file, h._source_sheet, h._source_row_num
),
h_prov1 AS (
  SELECT h.*,
         coalesce(pe.psgc_code, CASE WHEN pr.n = 1 THEN pr.psgc_code END,
                  CASE WHEN pr.n IS NULL AND pa.n = 1 THEN pa.psgc_code END) AS prov_psgc
  FROM h
  LEFT JOIN prov_region pr ON pr._source_key = h._source_key AND pr._source_file = h._source_file
       AND pr._source_sheet <=> h._source_sheet AND pr._source_row_num = h._source_row_num
  LEFT JOIN prov_any pa ON pa._source_key = h._source_key AND pa._source_file = h._source_file
       AND pa._source_sheet <=> h._source_sheet AND pa._source_row_num = h._source_row_num
  LEFT JOIN public_health.silver.facilities_area_exceptions pe
    ON pe.source_group = 'hfsrb' AND pe.match_type = 'province_label' AND pe.match_value = h.prov_norm
),
cand AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num, g.psgc_code,
         (CASE WHEN h.prov_psgc IS NOT NULL AND g.province_psgc = h.prov_psgc THEN 2 ELSE 0 END)
       + (CASE WHEN h.city_hint AND g.geo_level = 'city' THEN 1
               WHEN NOT h.city_hint AND g.geo_level = 'municipality' THEN 1 ELSE 0 END) AS score
  FROM h_prov1 h
  JOIN fac_geo g
    ON g.geo_level IN ('city', 'municipality', 'submunicipality')
   AND g.region_psgc = h.region_psgc AND g.place_norm = h.city_place
),
-- no candidate in the labelled region: the same name anywhere (labels such as
-- REGION 12 / Cotabato City, NCR / Imus City put the place in the wrong region)
cand_any AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num, g.psgc_code,
         (CASE WHEN h.prov_psgc IS NOT NULL AND g.province_psgc = h.prov_psgc THEN 2 ELSE 0 END)
       + (CASE WHEN h.city_hint AND g.geo_level = 'city' THEN 1
               WHEN NOT h.city_hint AND g.geo_level = 'municipality' THEN 1 ELSE 0 END) AS score
  FROM h_prov1 h
  JOIN fac_geo g
    ON g.geo_level IN ('city', 'municipality', 'submunicipality') AND g.place_norm = h.city_place
  WHERE NOT EXISTS (SELECT 1 FROM cand c WHERE c._source_key = h._source_key AND c._source_file = h._source_file
                      AND c._source_sheet <=> h._source_sheet AND c._source_row_num = h._source_row_num)
),
-- the province column names a city (e.g. province 'Davao City', city 'Talomo District'):
-- used only when the city label matched nothing anywhere
city_as_prov AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num,
         CASE WHEN count(*) = 1 THEN max(g.psgc_code) END AS psgc_code
  FROM h_prov1 h
  JOIN fac_geo g ON g.geo_level = 'city' AND g.place_norm = h.prov_place
                AND (g.region_psgc = h.region_psgc OR h.region_psgc IS NULL)
  WHERE h.prov_psgc IS NULL
  GROUP BY h._source_key, h._source_file, h._source_sheet, h._source_row_num
),
best_any AS (
  SELECT _source_key, _source_file, _source_sheet, _source_row_num,
         count_if(score = max_s) AS n_best, max_by(psgc_code, score) AS best_psgc,
         array_join(array_sort(collect_list(psgc_code)), ',') AS all_candidates
  FROM (SELECT *, max(score) OVER (PARTITION BY _source_key, _source_file, _source_sheet, _source_row_num) AS max_s FROM cand_any)
  GROUP BY _source_key, _source_file, _source_sheet, _source_row_num
),
best AS (
  SELECT _source_key, _source_file, _source_sheet, _source_row_num,
         max(score) AS best_score,
         count_if(score = max_s) AS n_best,
         max_by(psgc_code, score) AS best_psgc,
         array_join(array_sort(collect_list(psgc_code)), ',') AS all_candidates
  FROM (SELECT *, max(score) OVER (PARTITION BY _source_key, _source_file, _source_sheet, _source_row_num) AS max_s FROM cand)
  GROUP BY _source_key, _source_file, _source_sheet, _source_row_num
),
exc AS (
  SELECT * FROM public_health.silver.facilities_area_exceptions
  WHERE source_group = 'hfsrb' AND match_type = 'area_label'
),
hfsrb_geo AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num,
         coalesce(e1.psgc_code, e2.psgc_code,
                  CASE WHEN b.n_best = 1 THEN b.best_psgc END,
                  CASE WHEN b.n_best IS NULL AND ba.n_best = 1 THEN ba.best_psgc END,
                  CASE WHEN b.n_best IS NULL AND ba.n_best IS NULL THEN cp.psgc_code END) AS psgc_code,
         CAST(NULL AS STRING) AS barangay_psgc,
         CASE WHEN coalesce(e1.psgc_code, e2.psgc_code) IS NOT NULL THEN 'exception'
              WHEN b.n_best = 1 THEN 'name_level_parent'
              WHEN b.n_best IS NULL AND ba.n_best = 1 THEN 'name_level_other_region'
              WHEN b.n_best IS NULL AND ba.n_best IS NULL AND cp.psgc_code IS NOT NULL THEN 'city_in_province_column' END AS geo_resolution_method,
         CASE WHEN coalesce(e1.psgc_code, e2.psgc_code, CASE WHEN b.n_best = 1 THEN b.best_psgc END,
                            CASE WHEN b.n_best IS NULL AND ba.n_best = 1 THEN ba.best_psgc END,
                            CASE WHEN b.n_best IS NULL AND ba.n_best IS NULL THEN cp.psgc_code END) IS NULL
              THEN 1 ELSE 0 END AS geo_unresolved,
         CASE WHEN eu.match_value IS NOT NULL THEN concat('reviewed_unresolvable: ', coalesce(b.all_candidates, ba.all_candidates, 'no candidate'))
              ELSE coalesce(b.all_candidates, ba.all_candidates) END AS geo_candidates
  FROM h_prov1 h
  LEFT JOIN best b
    ON b._source_key = h._source_key AND b._source_file = h._source_file
   AND b._source_sheet <=> h._source_sheet AND b._source_row_num = h._source_row_num
  LEFT JOIN best_any ba
    ON ba._source_key = h._source_key AND ba._source_file = h._source_file
   AND ba._source_sheet <=> h._source_sheet AND ba._source_row_num = h._source_row_num
  LEFT JOIN city_as_prov cp
    ON cp._source_key = h._source_key AND cp._source_file = h._source_file
   AND cp._source_sheet <=> h._source_sheet AND cp._source_row_num = h._source_row_num
  LEFT JOIN exc e1 ON e1.match_value = h.city_norm AND e1.parent_label_norm = h.prov_norm
  LEFT JOIN exc e2 ON e2.match_value = h.city_norm AND e2.parent_label_norm IS NULL
  -- labels a reviewer could not place: quarantined without failing the gate
  LEFT JOIN public_health.silver.facilities_area_exceptions eu
    ON eu.source_group = 'hfsrb' AND eu.match_type = 'unresolvable'
   AND eu.match_value = h.city_norm AND eu.parent_label_norm <=> h.prov_norm
)
SELECT * FROM nhfr_geo
UNION ALL
SELECT * FROM hfsrb_geo;


-- =============================================================================
-- [4] identity (plan 2.2 - 2.6)
--   a. NHFR record: source_record_id = nhfr_id
--   b. HFSRB hospital rows sharing one nhfr_id (rule 2.4):
--        1 exact duplicate            -> duplicate_of the kept row
--        2 same licence or same name + municipality -> one facility, latest validity kept
--        3 different facilities       -> code not used for these rows (ambiguous_nhfr_code)
--   c. usable code that exists in NHFR -> joins that NHFR facility
--      (non-hospital lists: service_of_existing_facility, no second registry row)
--   d. no usable code: exact facility name + municipality against NHFR hospitals not
--      already matched (hospital list only)
--   e. overrides from facilities_id_map_override
--   f. everything else is its own facility: source_record_id = category | name | municipality
--   facility_key: reused from the previous id map for the same (_source_key, source_record_id);
--   an HFSRB record joined to NHFR takes the NHFR facility's key; otherwise a new uuid.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__records_pre AS
WITH d AS (
  SELECT c.*, g.psgc_code, g.barangay_psgc, g.geo_resolution_method, g.geo_unresolved, g.geo_candidates,
         fac_fname(c.facility_name) AS fname_norm
  FROM public_health.silver.facilities_registry_silver__classified c
  JOIN public_health.silver.facilities_registry_silver__geo g
    ON g._source_key = c._source_key AND g._source_file = c._source_file
   AND g._source_sheet <=> c._source_sheet AND g._source_row_num = c._source_row_num
  WHERE c.row_class = 'data'
),
nhfr AS (SELECT * FROM d WHERE _source_key = 'doh_nhfr'),
-- b. HFSRB hospital rows that share an nhfr_id
hh AS (
  SELECT d.*,
         count(*) OVER (PARTITION BY _source_key, nhfr_id) AS n_code,
         size(collect_set(coalesce(psgc_code, fac_norm(city_label))) OVER (PARTITION BY _source_key, nhfr_id)) AS n_places,
         size(collect_set(fname_norm) OVER (PARTITION BY _source_key, nhfr_id)) AS n_names,
         row_number() OVER (PARTITION BY _source_key, nhfr_id ORDER BY license_valid_until DESC NULLS LAST, _source_file, _source_sheet, _source_row_num) AS rn_code
  FROM d WHERE _source_key <> 'doh_nhfr' AND nhfr_id IS NOT NULL
),
hh_rule AS (
  SELECT h.*,
         CASE WHEN n_code = 1 THEN 'single'
              WHEN n_code > 1 AND n_places = 1 AND n_names = 1 THEN 'same_facility'
              ELSE 'ambiguous' END AS code_rule,
         first_value(struct(_source_file, _source_sheet, _source_row_num)) OVER (PARTITION BY _source_key, nhfr_id ORDER BY rn_code) AS kept_row
  FROM hh h
),
hfsrb AS (
  SELECT d.*,
         r.code_rule,
         CASE WHEN r.code_rule = 'same_facility' AND r.rn_code > 1 THEN r.kept_row END AS dup_kept_row,
         -- the id the row may use to join NHFR
         CASE WHEN d.nhfr_id IS NOT NULL AND coalesce(r.code_rule, 'single') <> 'ambiguous' THEN d.nhfr_id END AS usable_nhfr_id
  FROM d
  LEFT JOIN hh_rule r
    ON r._source_key = d._source_key AND r._source_file = d._source_file
   AND r._source_sheet <=> d._source_sheet AND r._source_row_num = d._source_row_num
  WHERE d._source_key <> 'doh_nhfr'
),
-- c. a code match is accepted only when the two records agree on the place (same LGU) or
--    share at least half of their distinctive name words; otherwise the code is not trusted
--    (code_name_place_mismatch) and the row goes on to the name match like a row without a code.
code_match AS (
  SELECT h.* EXCEPT (usable_nhfr_id),
         CASE WHEN n.nhfr_id IS NOT NULL AND NOT (fac_lgu(n.psgc_code) <=> fac_lgu(h.psgc_code))
                   AND coalesce(fac_name_similarity(n.facility_name, h.facility_name), 0) < 0.5
              THEN NULL ELSE h.usable_nhfr_id END AS usable_nhfr_id,
         CASE WHEN n.nhfr_id IS NOT NULL AND NOT (fac_lgu(n.psgc_code) <=> fac_lgu(h.psgc_code))
                   AND coalesce(fac_name_similarity(n.facility_name, h.facility_name), 0) < 0.5
              THEN NULL ELSE n.nhfr_id END AS matched_nhfr_id,
         (n.nhfr_id IS NOT NULL AND NOT (fac_lgu(n.psgc_code) <=> fac_lgu(h.psgc_code))
          AND coalesce(fac_name_similarity(n.facility_name, h.facility_name), 0) < 0.5) AS code_rejected
  FROM hfsrb h LEFT JOIN nhfr n ON n.nhfr_id = h.usable_nhfr_id
),
-- d. name + municipality match for hospital-list rows without a code match
name_cand AS (
  SELECT h._source_key, h._source_file, h._source_sheet, h._source_row_num, n.nhfr_id
  FROM code_match h
  JOIN nhfr n ON n.fname_norm = h.fname_norm AND n.psgc_code = h.psgc_code
  WHERE h._source_key = 'doh_hfsrb_hospitals' AND h.matched_nhfr_id IS NULL AND h.dup_kept_row IS NULL
    AND n.nhfr_id NOT IN (SELECT matched_nhfr_id FROM code_match
                          WHERE matched_nhfr_id IS NOT NULL AND _source_key = 'doh_hfsrb_hospitals' AND dup_kept_row IS NULL)
),
name_ok AS (       -- exactly one NHFR candidate, and that NHFR facility is claimed by only this row
  SELECT * FROM (
    SELECT c.*, count(*) OVER (PARTITION BY _source_key, _source_file, _source_sheet, _source_row_num) AS n_for_row,
                count(*) OVER (PARTITION BY nhfr_id) AS n_for_nhfr
    FROM name_cand c) WHERE n_for_row = 1 AND n_for_nhfr = 1
),
ov AS (
  SELECT * FROM public_health.silver.facilities_id_map_override
  WHERE coalesce(review_status, 'accepted') = 'accepted'
),
h_ident AS (
  SELECT h.*,
         CASE WHEN h._source_key = 'doh_hfsrb_hospitals' THEN coalesce(h.matched_nhfr_id, nm.nhfr_id) ELSE h.matched_nhfr_id END AS joined_nhfr_id,
         CASE WHEN h.matched_nhfr_id IS NOT NULL THEN 'nhfr_id'
              WHEN nm.nhfr_id IS NOT NULL THEN 'name_municipality' END AS auto_match_method,
         -- stable identifier of the source record
         CASE WHEN h.usable_nhfr_id IS NOT NULL THEN concat('nhfr:', h.usable_nhfr_id)
              WHEN h.psgc_code IS NOT NULL THEN concat('name:', h.facility_category, '|', h.fname_norm, '|', h.psgc_code)
         END AS source_record_id
  FROM code_match h
  LEFT JOIN name_ok nm
    ON nm._source_key = h._source_key AND nm._source_file = h._source_file
   AND nm._source_sheet <=> h._source_sheet AND nm._source_row_num = h._source_row_num
),
h_final AS (
  SELECT h.*,
         CASE WHEN o.action = 'link_to_nhfr' THEN CAST(o.nhfr_id AS BIGINT)
              WHEN o.action = 'own_facility' THEN NULL
              ELSE h.joined_nhfr_id END AS final_nhfr_id,
         CASE WHEN o.action IS NOT NULL THEN 'manual' ELSE h.auto_match_method END AS match_method
  FROM h_ident h
  LEFT JOIN ov o ON o._source_key = h._source_key AND o.source_record_id = h.source_record_id
),
-- duplicate source records among HFSRB-only rows (same id in one list): latest validity kept
h_dedup AS (
  SELECT h.*,
         row_number() OVER (PARTITION BY _source_key, source_record_id
                            ORDER BY license_valid_until DESC NULLS LAST, _source_file, _source_sheet, _source_row_num) AS rn_rec,
         first_value(struct(_source_file, _source_sheet, _source_row_num)) OVER (
           PARTITION BY _source_key, source_record_id
           ORDER BY license_valid_until DESC NULLS LAST, _source_file, _source_sheet, _source_row_num) AS rec_kept_row
  FROM h_final h
),
prev_map AS (SELECT * FROM public_health.silver.facilities_id_map_silver),
nhfr_keyed AS (
  SELECT n.*, coalesce(p.facility_key, uuid()) AS facility_key, p._first_build_id AS prev_first_build_id
  FROM nhfr n
  LEFT JOIN prev_map p ON p._source_key = 'doh_nhfr' AND p.source_record_id = CAST(n.nhfr_id AS STRING)
)
SELECT n.input_table, n._source_key, n._source_file, n._source_sheet, n._source_row_num, n._ingested_at, n._as_of_date,
       n.facility_category, n.source_id_raw, n.nhfr_id, CAST(n.nhfr_id AS STRING) AS source_record_id,
       true AS code_validated,
       'data' AS record_class, CAST(NULL AS STRING) AS reason,
       'matched' AS match_status, 'nhfr_id' AS match_method, n.nhfr_id AS joined_nhfr_id,
       CAST(NULL AS STRING) AS kept_file, CAST(NULL AS STRING) AS kept_sheet, CAST(NULL AS INT) AS kept_row,
       n.facility_key, n.prev_first_build_id,
       n.facility_name, n.ownership, n.ownership_class, n.service_level, n.nhfr_reported_beds, n.hfsrb_abc,
       n.dialysis_stations, n.blood_service_type, n.nhfr_licensing_status, n.license_valid_until, n.license_no,
       n.psgc_code, n.barangay_psgc, n.geo_resolution_method, n.geo_unresolved, n.geo_candidates,
       n.city_label AS geo_name_source, n.value_columns, n.code_issue
FROM nhfr_keyed n
UNION ALL
SELECT h.input_table, h._source_key, h._source_file, h._source_sheet, h._source_row_num, h._ingested_at, h._as_of_date,
       h.facility_category, h.source_id_raw, h.nhfr_id, h.source_record_id,
       -- a dialysis code cannot be checked: the NHFR export holds hospitals only (plan 2.2)
       CASE WHEN h.facility_category = 'dialysis_clinic' THEN false ELSE (h.nhfr_id IS NOT NULL) END AS code_validated,
       CASE WHEN h.code_rule = 'same_facility' AND h.dup_kept_row IS NOT NULL THEN 'duplicate'
            WHEN h.final_nhfr_id IS NULL AND h.psgc_code IS NULL THEN 'quarantined'
            WHEN h.rn_rec > 1 THEN 'duplicate'
            ELSE 'data' END AS record_class,
       CASE WHEN h.code_rule = 'same_facility' AND h.dup_kept_row IS NOT NULL THEN 'same_facility_older_record'
            WHEN h.final_nhfr_id IS NULL AND h.psgc_code IS NULL
                 THEN CASE WHEN h.city_label IS NULL THEN 'no_place_label'
                           WHEN h.geo_candidates LIKE 'reviewed_unresolvable%' THEN 'place_reviewed_unresolvable'
                           ELSE 'geo_unresolved' END
            WHEN h.rn_rec > 1 THEN 'same_record_in_list' END AS reason,
       CASE WHEN h.final_nhfr_id IS NOT NULL AND h.facility_category <> 'hospital' THEN 'service_of_existing_facility'
            WHEN h.final_nhfr_id IS NOT NULL THEN 'matched'
            WHEN h.code_rule = 'ambiguous' THEN 'ambiguous_code_own_facility'
            ELSE 'new_facility' END AS match_status,
       coalesce(h.match_method, 'new') AS match_method,
       h.final_nhfr_id AS joined_nhfr_id,
       coalesce(h.dup_kept_row._source_file, CASE WHEN h.rn_rec > 1 THEN h.rec_kept_row._source_file END),
       coalesce(h.dup_kept_row._source_sheet, CASE WHEN h.rn_rec > 1 THEN h.rec_kept_row._source_sheet END),
       coalesce(h.dup_kept_row._source_row_num, CASE WHEN h.rn_rec > 1 THEN h.rec_kept_row._source_row_num END),
       -- joined to NHFR: the NHFR facility's key. Own facility: the key it had before, but only
       -- if it was its own facility then too (a record that used to join NHFR gets a new key)
       CASE WHEN h.final_nhfr_id IS NOT NULL THEN nk.facility_key
            WHEN p.match_status IN ('new_facility', 'ambiguous_code_own_facility') THEN p.facility_key
       END AS facility_key_pre,                                           -- new uuid assigned below
       p._first_build_id,
       h.facility_name, h.ownership, h.ownership_class, h.service_level, h.nhfr_reported_beds, h.hfsrb_abc,
       h.dialysis_stations, h.blood_service_type, h.nhfr_licensing_status, h.license_valid_until, h.license_no,
       h.psgc_code, h.barangay_psgc, h.geo_resolution_method, h.geo_unresolved, h.geo_candidates,
       h.city_label, h.value_columns,
       coalesce(h.code_issue, CASE WHEN h.code_rule = 'ambiguous' THEN 'ambiguous_nhfr_code'
                                   WHEN h.code_rejected THEN 'code_name_place_mismatch' END)
FROM h_dedup h
LEFT JOIN nhfr_keyed nk ON nk.nhfr_id = h.final_nhfr_id
LEFT JOIN prev_map p ON p._source_key = h._source_key AND p.source_record_id = h.source_record_id;

-- one new uuid per new HFSRB-only facility (one per source record, shared by its duplicates)
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__records AS
WITH new_keys AS (
  SELECT _source_key, source_record_id, uuid() AS new_key
  FROM (SELECT DISTINCT _source_key, source_record_id
        FROM public_health.silver.facilities_registry_silver__records_pre
        WHERE facility_key IS NULL AND source_record_id IS NOT NULL)
)
SELECT r.* EXCEPT (facility_key), coalesce(r.facility_key, k.new_key) AS facility_key
FROM public_health.silver.facilities_registry_silver__records_pre r
LEFT JOIN new_keys k USING (_source_key, source_record_id);


-- =============================================================================
-- [5] candidates
-- =============================================================================

-- [5a] id map: one row per (_source_key, source_record_id) seen now (the record kept for it),
--      plus rows of the previous version not seen now, carried unchanged (old build id).
CREATE OR REPLACE TABLE public_health.silver.facilities_id_map_silver__candidate AS
WITH cur AS (
  SELECT _source_key, source_record_id, source_id_raw, nhfr_id AS source_id_norm, code_validated, facility_key,
         match_method, match_status,
         coalesce(prev_first_build_id, idmap_build_id) AS _first_build_id,
         idmap_build_id AS _silver_build_id,
         _source_file, _source_sheet, _source_row_num, _as_of_date
  FROM public_health.silver.facilities_registry_silver__records
  WHERE record_class = 'data'
)
SELECT * FROM cur
UNION ALL
SELECT p.* FROM public_health.silver.facilities_id_map_silver p
LEFT ANTI JOIN cur c ON c._source_key = p._source_key AND c.source_record_id = p.source_record_id;

-- [5b] registry
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__candidate AS
WITH n AS (SELECT * FROM public_health.silver.facilities_registry_silver__records
           WHERE _source_key = 'doh_nhfr' AND record_class = 'data'),
     h AS (SELECT * FROM public_health.silver.facilities_registry_silver__records
           WHERE _source_key <> 'doh_nhfr' AND record_class = 'data'
             AND match_status <> 'service_of_existing_facility'),
facilities AS (
  -- NHFR facilities, with their HFSRB hospital record when one joined
  SELECT n.facility_key, n.facility_category, struct(n.*) AS nr, struct(hj.*) AS hr
  FROM n LEFT JOIN h hj ON hj.joined_nhfr_id = n.nhfr_id AND hj._source_key = 'doh_hfsrb_hospitals'
  UNION ALL
  -- HFSRB facilities that did not join an NHFR facility (the NHFR struct is all nulls)
  SELECT h.facility_key, h.facility_category, struct(n.*) AS nr, struct(h.*) AS hr
  FROM h LEFT JOIN n ON false
  WHERE h.joined_nhfr_id IS NULL
),
cur AS (
  SELECT f.facility_key,
         f.facility_category,
         coalesce(nr.facility_name, hr.facility_name)                     AS facility_name,
         nr.nhfr_id                                                       AS nhfr_id,
         hr.license_no                                                    AS license_no,
         coalesce(nr.ownership, hr.ownership)                             AS ownership,
         coalesce(nr.ownership_class, hr.ownership_class)                 AS ownership_class,
         coalesce(nr.service_level, hr.service_level)                     AS service_level,
         concat('psgc:', coalesce(nr.psgc_code, hr.psgc_code))            AS area_key,
         coalesce(nr.psgc_code, hr.psgc_code)                             AS psgc_code,
         nr.barangay_psgc                                                 AS barangay_psgc,
         coalesce(nr.geo_name_source, hr.geo_name_source)                 AS geo_name_source,
         CASE WHEN nr.facility_key IS NOT NULL THEN nr.geo_resolution_method ELSE hr.geo_resolution_method END AS geo_resolution_method,
         (nr.facility_key IS NOT NULL AND hr.facility_key IS NOT NULL AND hr.psgc_code IS NOT NULL
          AND fac_lgu(hr.psgc_code) <> fac_lgu(nr.psgc_code))             AS geo_conflict_nhfr_hfsrb,
         nr.nhfr_reported_beds,
         hr.hfsrb_abc,
         hr.hfsrb_abc                                                     AS authorized_beds,
         coalesce(hr.hfsrb_abc, nr.nhfr_reported_beds)                    AS analysis_beds,
         CASE WHEN hr.hfsrb_abc IS NOT NULL THEN 'authorized'
              WHEN nr.nhfr_reported_beds IS NOT NULL THEN 'nhfr_reported' END AS analysis_beds_basis,
         hr.dialysis_stations,
         CASE WHEN f.facility_category = 'hospital' AND hr.facility_key IS NOT NULL THEN hr.blood_service_type IS NOT NULL END AS has_blood_service,
         hr.blood_service_type,
         nr.nhfr_licensing_status,
         hr.license_valid_until,
         hr.license_valid_until >= hr._as_of_date                         AS is_license_valid_on_as_of,
         (nr.facility_key IS NOT NULL)                                    AS in_nhfr,
         (hr.facility_key IS NOT NULL)                                    AS in_hfsrb,
         CASE WHEN nr.facility_key IS NOT NULL AND hr.facility_key IS NOT NULL THEN hr.match_method
              WHEN nr.facility_key IS NOT NULL THEN 'nhfr_only' ELSE 'hfsrb_only' END AS match_method,
         (nr.facility_key IS NOT NULL)                                    AS is_currently_listed_nhfr,
         (hr.facility_key IS NOT NULL)                                    AS is_currently_listed_hfsrb,
         'listed'                                                         AS facility_status,
         registry_build_id                                                AS _silver_build_id,
         nr._source_key AS _nhfr_source_key, nr._source_file AS _nhfr_source_file,
         nr._source_row_num AS _nhfr_source_row_num, nr._as_of_date AS _nhfr_as_of_date,
         hr._source_key AS _hfsrb_source_key, hr._source_file AS _hfsrb_source_file, hr._source_sheet AS _hfsrb_source_sheet,
         hr._source_row_num AS _hfsrb_source_row_num, hr._as_of_date AS _hfsrb_as_of_date,
         -- silver column -> bronze column, only for the side each value came from (metric_groups in the plan)
         map_concat(
           CASE WHEN nr.facility_key IS NOT NULL
                THEN map('nhfr_reported_beds', 'Bed_Capacity', 'nhfr_licensing_status', 'Licensing_Status')
                ELSE CAST(map() AS MAP<STRING, STRING>) END,
           CASE WHEN hr.facility_key IS NOT NULL THEN map_filter(
                  map('hfsrb_abc', hr.value_columns['hfsrb_abc'], 'dialysis_stations', hr.value_columns['dialysis_stations'],
                      'license_valid_until', hr.value_columns['license_valid_until'],
                      'blood_service_type', hr.value_columns['blood_service_type']),
                  (k, v) -> v IS NOT NULL)
                ELSE CAST(map() AS MAP<STRING, STRING>) END,
           -- columns filled from either side: the side that supplied the value
           map('facility_name', CASE WHEN nr.facility_name IS NOT NULL THEN 'nhfr:Facility_Name' ELSE 'hfsrb:facility_name' END,
               'ownership', CASE WHEN nr.ownership IS NOT NULL THEN 'nhfr:Ownership_Major_Classification' ELSE 'hfsrb:ownership' END),
           CASE WHEN coalesce(nr.service_level, hr.service_level) IS NOT NULL
                THEN map('service_level', CASE WHEN nr.service_level IS NOT NULL THEN 'nhfr:Service_Capability' ELSE 'hfsrb:service_capability' END)
                ELSE CAST(map() AS MAP<STRING, STRING>) END
         ) AS _metric_source_columns
  FROM facilities f
),
hashed AS (
  SELECT c.*,
         sha2(to_json(named_struct(
           'facility_category', facility_category, 'facility_name', facility_name, 'nhfr_id', nhfr_id,
           'license_no', license_no, 'ownership', ownership, 'ownership_class', ownership_class,
           'service_level', service_level, 'psgc_code', psgc_code, 'barangay_psgc', barangay_psgc,
           'nhfr_reported_beds', nhfr_reported_beds, 'hfsrb_abc', hfsrb_abc, 'dialysis_stations', dialysis_stations,
           'blood_service_type', blood_service_type, 'nhfr_licensing_status', nhfr_licensing_status,
           'license_valid_until', license_valid_until, 'in_nhfr', in_nhfr, 'in_hfsrb', in_hfsrb)), 256) AS _row_hash
  FROM cur c
),
prev AS (SELECT * FROM public_health.silver.facilities_registry_silver)
SELECT h.facility_key, h.facility_category, h.facility_name, h.nhfr_id, h.license_no, h.ownership, h.ownership_class,
       h.service_level, h.area_key, h.psgc_code, h.barangay_psgc, h.geo_name_source, h.geo_resolution_method,
       h.geo_conflict_nhfr_hfsrb, h.nhfr_reported_beds, h.hfsrb_abc, h.authorized_beds, h.analysis_beds,
       h.analysis_beds_basis, h.dialysis_stations, h.has_blood_service, h.blood_service_type, h.nhfr_licensing_status,
       h.license_valid_until, h.is_license_valid_on_as_of, h.in_nhfr, h.in_hfsrb, h.match_method,
       h.is_currently_listed_nhfr, h.is_currently_listed_hfsrb, h.facility_status,
       h._row_hash,
       coalesce(p._first_build_id, registry_build_id)                                   AS _first_build_id,
       CASE WHEN p._row_hash <=> h._row_hash THEN p._business_change_build_id ELSE registry_build_id END AS _business_change_build_id,
       CAST(NULL AS STRING)                                                             AS _delisted_build_id,
       h._silver_build_id,
       h._nhfr_source_key, h._nhfr_source_file, h._nhfr_source_row_num, h._nhfr_as_of_date,
       h._hfsrb_source_key, h._hfsrb_source_file, h._hfsrb_source_sheet, h._hfsrb_source_row_num, h._hfsrb_as_of_date,
       h._metric_source_columns
FROM hashed h
LEFT JOIN prev p ON p.facility_key = h.facility_key
UNION ALL
-- facilities no source lists any more: kept unchanged, so their lineage still points at the build
-- that last read them; only the listing flags and _delisted_build_id change (plan 5.3).
-- The declared-inputs gate in [6] guarantees this build read every source completely.
SELECT p.facility_key, p.facility_category, p.facility_name, p.nhfr_id, p.license_no, p.ownership, p.ownership_class,
       p.service_level, p.area_key, p.psgc_code, p.barangay_psgc, p.geo_name_source, p.geo_resolution_method,
       p.geo_conflict_nhfr_hfsrb, p.nhfr_reported_beds, p.hfsrb_abc, p.authorized_beds, p.analysis_beds,
       p.analysis_beds_basis, p.dialysis_stations, p.has_blood_service, p.blood_service_type, p.nhfr_licensing_status,
       p.license_valid_until, p.is_license_valid_on_as_of, p.in_nhfr, p.in_hfsrb, p.match_method,
       false, false, 'not_listed',
       p._row_hash, p._first_build_id, p._business_change_build_id,
       coalesce(p._delisted_build_id, registry_build_id),
       p._silver_build_id,
       p._nhfr_source_key, p._nhfr_source_file, p._nhfr_source_row_num, p._nhfr_as_of_date,
       p._hfsrb_source_key, p._hfsrb_source_file, p._hfsrb_source_sheet, p._hfsrb_source_row_num, p._hfsrb_as_of_date,
       p._metric_source_columns
FROM prev p
LEFT ANTI JOIN hashed h ON h.facility_key = p.facility_key;


-- =============================================================================
-- [5c] dispositions (lineage-qc-contract.md v1.3, section 7.1): one row per bronze row that was
--      NOT loaded as itself. Loaded rows are not stored: the silver rows carry their bronze row
--      reference, and rows loaded per input = input_rows - rows here (quality.v_build_reconciliation).
--      merged_into rows are stored (with the facility_key) but count as loaded: they contributed.
--      A service record of an existing hospital is quarantined (target_key = that hospital's key).
--      Saved before the gates, so the gates check exactly what step [10] writes.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__dispositions AS
SELECT * FROM (
SELECT registry_build_id AS build_id, 'public_health.silver.facilities_registry_silver' AS target_table,
       c._source_key AS source_key, c._source_file AS source_file, c._source_sheet AS source_sheet,
       c._source_row_num AS source_row_num, i.load_batch_id, i.content_checksum,
       CASE WHEN c.row_class = 'non_data' THEN 'dropped_non_data'
            WHEN r.record_class = 'duplicate' THEN 'duplicate_of'
            WHEN r.record_class = 'quarantined' THEN 'quarantined'
            -- a second licence (infirmary, primary care, dialysis) of a hospital already in the registry:
            -- it adds no value to the hospital row, so it is not loaded; kept for review (plan 2.1)
            WHEN r.match_status = 'service_of_existing_facility' THEN 'quarantined'
            WHEN r._source_key <> 'doh_nhfr' AND r.joined_nhfr_id IS NOT NULL THEN 'merged_into'
            ELSE 'loaded' END AS disposition,
       CASE WHEN c.row_class = 'non_data' THEN c.reason
            WHEN r.record_class IN ('duplicate', 'quarantined') THEN r.reason
            WHEN r.match_status = 'service_of_existing_facility' THEN r.match_status
            WHEN r._source_key <> 'doh_nhfr' AND r.joined_nhfr_id IS NOT NULL THEN r.match_status END AS reason,
       CASE WHEN r.record_class = 'duplicate' THEN to_json(named_struct(
              '_source_key', r._source_key, '_source_file', r.kept_file, '_source_sheet', r.kept_sheet,
              '_source_row_num', r.kept_row, '_content_checksum', i.content_checksum))
            WHEN r.record_class = 'data' THEN to_json(named_struct('facility_key', r.facility_key)) END AS target_key,
       current_timestamp() AS recorded_at
FROM public_health.silver.facilities_registry_silver__classified c
LEFT JOIN public_health.silver.facilities_registry_silver__records r
  ON r._source_key = c._source_key AND r._source_file = c._source_file
 AND r._source_sheet <=> c._source_sheet AND r._source_row_num = c._source_row_num
LEFT JOIN public_health.silver.facilities_registry_silver__inputs i
  ON i.input_kind = 'bronze_source' AND i.source_key = c._source_key AND i.source_file = c._source_file
 AND i.source_sheet <=> c._source_sheet
) d
WHERE disposition <> 'loaded';


-- =============================================================================
-- [6] gates and output validation: every row must have failures = 0
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver__gates AS
SELECT 'inputs: declared file or sheet not read' AS gate, count(*) AS failures
  FROM fac_declared_inputs d
  LEFT ANTI JOIN public_health.silver.facilities_registry_silver__inputs i
    ON i.input_kind = 'bronze_source' AND i.source_key = d.source_key AND i.source_sheet <=> d.source_sheet
UNION ALL
SELECT 'inputs: file or sheet read that is not declared', count(*)
  FROM public_health.silver.facilities_registry_silver__inputs i
  LEFT ANTI JOIN fac_declared_inputs d ON i.source_key = d.source_key AND i.source_sheet <=> d.source_sheet
  WHERE i.input_kind = 'bronze_source'
UNION ALL
SELECT 'inputs: file or sheet with more than one load time', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_registry_silver__inputs WHERE input_kind = 'bronze_source'
  GROUP BY input_table, source_key, source_file, source_sheet HAVING count(*) > 1)
UNION ALL
SELECT 'inputs: more than one ingestion log match', count(*)
  FROM public_health.silver.facilities_registry_silver__inputs WHERE input_kind = 'bronze_source' AND metadata_matches > 1
UNION ALL
SELECT 'inputs: no processed ingestion log match (strict_load_lookup = true)', count(*)
  FROM public_health.silver.facilities_registry_silver__inputs
  WHERE input_kind = 'bronze_source' AND metadata_matches = 0 AND strict_load_lookup
UNION ALL
SELECT 'inputs: geography snapshot is empty', CASE WHEN (SELECT count(*) FROM fac_geo) = 0 THEN 1 ELSE 0 END
UNION ALL
SELECT 'NHFR: code not in the format DOH + 15 digits', count(*)
  FROM public_health.silver.facilities_registry_silver__classified WHERE _source_key = 'doh_nhfr' AND row_class = 'data' AND nhfr_id IS NULL
UNION ALL
SELECT 'NHFR: nhfr_id not unique', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_registry_silver__classified WHERE _source_key = 'doh_nhfr' AND row_class = 'data'
  GROUP BY nhfr_id HAVING count(*) > 1)
UNION ALL
SELECT 'NHFR: city/municipality code not in geography (after the code exceptions)', count(*)
  FROM public_health.silver.facilities_registry_silver__geo WHERE _source_key = 'doh_nhfr' AND geo_unresolved = 1
UNION ALL
SELECT 'HFSRB: region label not recognised', count(*)
  FROM public_health.silver.facilities_registry_silver__classified
  WHERE _source_key <> 'doh_nhfr' AND row_class = 'data' AND fac_region_psgc(region_label) IS NULL
UNION ALL
SELECT 'HFSRB: record without an NHFR match whose place does not resolve (add an area exception)', count(*)
  FROM public_health.silver.facilities_registry_silver__records WHERE record_class = 'quarantined' AND reason = 'geo_unresolved'
UNION ALL
SELECT 'HFSRB hospitals: an nhfr_id joined by more than one record (rule 2.4)', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_registry_silver__records
  WHERE _source_key = 'doh_hfsrb_hospitals' AND record_class = 'data' AND joined_nhfr_id IS NOT NULL
  GROUP BY joined_nhfr_id HAVING count(*) > 1)
UNION ALL
SELECT 'bronze row classified more than once', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_registry_silver__classified
  GROUP BY _source_key, _source_file, _source_sheet, _source_row_num HAVING count(*) > 1)
UNION ALL
SELECT 'data row without an identity decision', count(*)
  FROM public_health.silver.facilities_registry_silver__classified c
  LEFT ANTI JOIN public_health.silver.facilities_registry_silver__records r
    ON r._source_key = c._source_key AND r._source_file = c._source_file
   AND r._source_sheet <=> c._source_sheet AND r._source_row_num = c._source_row_num
  WHERE c.row_class = 'data'
UNION ALL
SELECT 'id map: key not unique', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_id_map_silver__candidate GROUP BY _source_key, source_record_id HAVING count(*) > 1)
UNION ALL
SELECT 'id map: facility_key null', count(*) FROM public_health.silver.facilities_id_map_silver__candidate WHERE facility_key IS NULL
UNION ALL
SELECT 'registry: facility_key null or not unique', count(*) FROM (
  SELECT facility_key FROM public_health.silver.facilities_registry_silver__candidate
  GROUP BY facility_key HAVING count(*) > 1 OR facility_key IS NULL)
UNION ALL
SELECT 'registry: listed facility with an area not in geography', count(*)
  FROM public_health.silver.facilities_registry_silver__candidate r
  WHERE r.facility_status = 'listed'
    AND NOT EXISTS (SELECT 1 FROM fac_geo g WHERE g.psgc_code = r.psgc_code)
UNION ALL
SELECT 'registry: every registry key is in the id map', count(*)
  FROM public_health.silver.facilities_registry_silver__candidate r
  LEFT ANTI JOIN public_health.silver.facilities_id_map_silver__candidate m ON m.facility_key = r.facility_key
UNION ALL
-- lineage requirements (plan 5.4): a group is filled exactly when its side contributed
SELECT 'lineage: nhfr group does not match in_nhfr', count(*)
  FROM public_health.silver.facilities_registry_silver__candidate
  WHERE in_nhfr <> (_nhfr_source_key IS NOT NULL AND _nhfr_source_file IS NOT NULL
                    AND _nhfr_source_row_num IS NOT NULL AND _nhfr_as_of_date IS NOT NULL)
     OR (NOT in_nhfr AND coalesce(_nhfr_source_key, _nhfr_source_file, CAST(_nhfr_source_row_num AS STRING), CAST(_nhfr_as_of_date AS STRING)) IS NOT NULL)
UNION ALL
SELECT 'lineage: hfsrb group does not match in_hfsrb', count(*)
  FROM public_health.silver.facilities_registry_silver__candidate
  WHERE in_hfsrb <> (_hfsrb_source_key IS NOT NULL AND _hfsrb_source_file IS NOT NULL AND _hfsrb_source_sheet IS NOT NULL
                     AND _hfsrb_source_row_num IS NOT NULL AND _hfsrb_as_of_date IS NOT NULL)
     OR (NOT in_hfsrb AND coalesce(_hfsrb_source_key, _hfsrb_source_file, _hfsrb_source_sheet,
                                   CAST(_hfsrb_source_row_num AS STRING), CAST(_hfsrb_as_of_date AS STRING)) IS NOT NULL)
UNION ALL
SELECT 'lineage: _silver_build_id or _metric_source_columns null', count(*)
  FROM public_health.silver.facilities_registry_silver__candidate WHERE _silver_build_id IS NULL OR _metric_source_columns IS NULL
UNION ALL
SELECT 'lineage: _metric_source_columns key without a metric group', count(*)
  FROM (SELECT explode(map_keys(_metric_source_columns)) AS k FROM public_health.silver.facilities_registry_silver__candidate
        WHERE _silver_build_id = registry_build_id)
  WHERE k NOT IN ('nhfr_reported_beds', 'nhfr_licensing_status', 'hfsrb_abc', 'dialysis_stations',
                  'license_valid_until', 'blood_service_type', 'facility_name', 'ownership', 'service_level')
UNION ALL
-- output validation
SELECT 'output: hospitals <> NHFR hospitals + HFSRB hospital records that joined no NHFR facility',
       abs((SELECT count(*) FROM public_health.silver.facilities_registry_silver__candidate
            WHERE facility_category = 'hospital' AND _silver_build_id = registry_build_id)
         - (SELECT count(*) FROM public_health.silver.facilities_registry_silver__records WHERE _source_key = 'doh_nhfr' AND record_class = 'data')
         - (SELECT count(*) FROM public_health.silver.facilities_registry_silver__records
            WHERE _source_key = 'doh_hfsrb_hospitals' AND record_class = 'data' AND joined_nhfr_id IS NULL))
UNION ALL
SELECT 'control: NHFR bed capacity = 119,862 (bronze QC control)',
       CASE WHEN (SELECT sum(nhfr_reported_beds) FROM public_health.silver.facilities_registry_silver__candidate
                  WHERE in_nhfr AND _silver_build_id = registry_build_id) = 119862 THEN 0 ELSE 1 END
UNION ALL
SELECT 'control: HFSRB hospital ABC in silver = bronze ABC of the records kept',
       abs(coalesce((SELECT sum(hfsrb_abc) FROM public_health.silver.facilities_registry_silver__candidate
                     WHERE _hfsrb_source_key = 'doh_hfsrb_hospitals' AND _silver_build_id = registry_build_id), 0)
         - coalesce((SELECT sum(hfsrb_abc) FROM public_health.silver.facilities_registry_silver__records
                     WHERE _source_key = 'doh_hfsrb_hospitals' AND record_class = 'data'), 0))
UNION ALL
SELECT 'dispositions: classified rows <> bronze rows read', abs(
         (SELECT count(*) FROM public_health.silver.facilities_registry_silver__classified)
       - (SELECT sum(input_rows) FROM public_health.silver.facilities_registry_silver__inputs WHERE input_kind = 'bronze_source'))
UNION ALL
-- contract 7.1, per input: distinct bronze rows referenced by the registry (either side, this build)
--   = rows read - rows not loaded. merged_into counts as loaded (it contributed to a facility row).
SELECT 'reconciliation: inputs where rows referenced by silver <> rows read - rows not loaded', count(*)
FROM public_health.silver.facilities_registry_silver__inputs i
LEFT JOIN (
  SELECT k, f, s, count(*) AS n FROM (
    SELECT DISTINCT _nhfr_source_key AS k, _nhfr_source_file AS f, CAST(NULL AS STRING) AS s, _nhfr_source_row_num AS r
    FROM public_health.silver.facilities_registry_silver__candidate WHERE in_nhfr AND _silver_build_id = registry_build_id
    UNION
    SELECT DISTINCT _hfsrb_source_key, _hfsrb_source_file, _hfsrb_source_sheet, _hfsrb_source_row_num
    FROM public_health.silver.facilities_registry_silver__candidate WHERE in_hfsrb AND _silver_build_id = registry_build_id)
  GROUP BY k, f, s) ref
  ON ref.k = i.source_key AND ref.f = i.source_file AND ref.s <=> i.source_sheet
LEFT JOIN (
  SELECT source_key, source_file, source_sheet, count_if(disposition <> 'merged_into') AS n
  FROM public_health.silver.facilities_registry_silver__dispositions GROUP BY source_key, source_file, source_sheet) nl
  ON nl.source_key = i.source_key AND nl.source_file = i.source_file AND nl.source_sheet <=> i.source_sheet
WHERE i.input_kind = 'bronze_source' AND i.input_rows - coalesce(nl.n, 0) <> coalesce(ref.n, 0);


-- =============================================================================
-- [7] build log rows (status 'succeeded' only when every gate is 0)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_registry_build_log AS
WITH g AS (
  SELECT count_if(failures <> 0) AS failed_gates,
         concat_ws(' | ', collect_list(CASE WHEN failures <> 0 THEN concat(gate, ' = ', failures) END)) AS failed_names
  FROM public_health.silver.facilities_registry_silver__gates
)
SELECT b.build_id, 'silver' AS layer, 'facilities' AS domain, b.target_table,
       silver_build_started_at AS started_at, current_timestamp() AS finished_at,
       CASE WHEN g.failed_gates = 0 THEN 'succeeded' ELSE 'failed' END AS status,
       CASE WHEN g.failed_gates = 0 THEN b.n END AS row_count,
       CAST(NULL AS BIGINT) AS target_table_version,
       code_version AS code_version,
       to_json(named_struct('strict_load_lookup', strict_load_lookup, 'geography_checksum', geography_checksum,
                            'companion_build_id', b.companion)) AS parameters,
       job_run_id AS run_id,
       CASE WHEN g.failed_gates > 0 THEN substr(concat('gates failed: ', g.failed_names), 1, 500) END AS error_message
FROM (SELECT registry_build_id AS build_id, 'public_health.silver.facilities_registry_silver' AS target_table,
             (SELECT count(*) FROM public_health.silver.facilities_registry_silver__candidate) AS n, idmap_build_id AS companion
      UNION ALL
      SELECT idmap_build_id, 'public_health.silver.facilities_id_map_silver',
             (SELECT count(*) FROM public_health.silver.facilities_id_map_silver__candidate), registry_build_id) b
CROSS JOIN g;


-- =============================================================================
-- [8] stop here if a gate failed. The failed builds are logged; the published tables stay as
--     they were. Staging tables are kept for inspection:
--       SELECT * FROM public_health.silver.facilities_registry_silver__gates WHERE failures <> 0
-- =============================================================================
INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
       target_table_version, code_version, parameters, run_id, error_message
FROM fac_registry_build_log WHERE status = 'failed';

SELECT raise_error(concat('facilities registry was NOT published. ', error_message))
FROM fac_registry_build_log WHERE status = 'failed' AND target_table LIKE '%registry%';


-- =============================================================================
-- [9] write both tables from the validated candidates
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_id_map_silver AS
SELECT * FROM public_health.silver.facilities_id_map_silver__candidate;

CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver AS
SELECT * FROM public_health.silver.facilities_registry_silver__candidate;


-- =============================================================================
-- [10] record the builds
-- =============================================================================
INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
       -- Delta version just written. If your runtime rejects these lines, use CAST(NULL AS BIGINT).
       CASE WHEN target_table LIKE '%registry%'
            THEN (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_registry_silver))
            ELSE (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_id_map_silver)) END,
       code_version, parameters, run_id, error_message
FROM fac_registry_build_log;

-- the registry's inputs; the id map build read the same inputs
INSERT INTO public_health.quality.build_inputs
  (build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
   source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version)
SELECT build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
       source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version
FROM public_health.silver.facilities_registry_silver__inputs
UNION ALL
SELECT idmap_build_id, 'public_health.silver.facilities_id_map_silver', input_kind, input_table, input_table_version,
       source_key, source_file, source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version
FROM public_health.silver.facilities_registry_silver__inputs
WHERE NOT (input_kind = 'table' AND input_table LIKE '%facilities_registry_silver');

-- bronze rows not loaded as themselves (staged in [5c]); loaded rows are not written (contract 7.1)
INSERT INTO public_health.quality.row_disposition
  (build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
   content_checksum, disposition, reason, target_key, recorded_at)
SELECT build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
       content_checksum, disposition, reason, target_key, recorded_at
FROM public_health.silver.facilities_registry_silver__dispositions;


-- =============================================================================
-- [11] summary, then remove the staging tables (__classified and __records are kept until the
--      next run: they show how every bronze row was parsed and matched)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_registry_summary AS
SELECT facility_category, facility_status, match_method, count(*) AS facilities,
       sum(analysis_beds) AS analysis_beds, count_if(geo_conflict_nhfr_hfsrb) AS geo_conflicts
FROM public_health.silver.facilities_registry_silver
GROUP BY facility_category, facility_status, match_method;

DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__candidate;
DROP TABLE IF EXISTS public_health.silver.facilities_id_map_silver__candidate;
DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__gates;
DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__inputs;
DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__geo;
DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__records_pre;
DROP TABLE IF EXISTS public_health.silver.facilities_registry_silver__dispositions;

SELECT * FROM fac_registry_summary ORDER BY facility_category, facility_status, match_method;

In [0]:
-- =============================================================================
-- facilities_silver_02_metric.sql
-- Builds public_health.silver.facilities_metric_silver:
--   counts of facilities, beds and barangays for one area in one year, by ownership, from one
--   source. Wide layout: one column per measure (plan 4.4).
--   Key: (area_key, year, ownership, _source_key)
-- Follows facilities-silver-plan.md (sections 4, 5, 6) and lineage-qc-contract.md v1.2.
--
-- Sources (7 bronze tables, 15 files or sheets)
--   psa_hospitals_region_annual          PSA 9.18, hospitals by region and ownership, 2020-2024
--   psa_beds_region_annual               PSA 9.17, authorized beds by region and ownership, 2020-2024
--   psa_hospitals_national_annual        PSA OpenSTAT, hospitals by ownership, national, 2006-2015
--   psa_hospitals_beds_national_annual   PSA 9.16, hospitals and beds by ownership, national, 1990-2015
--   psa_hospitals_region_annual_legacy   PSA 9.17 (PSY 2024), hospitals by region and ownership, 2005-2015
--   psa_bhs_region_annual                PSA 9.19, barangay health stations by region, 2006-2023
--   doh_fhsis_demographic                FHSIS BGY & BHS tab, region / province / city, 2018-2025
-- Sources overlap on purpose (e.g. national hospitals 2006-2015 in three tables; BHS 2018-2023
-- in PSA 9.19 and FHSIS). Every source keeps its own rows; gold chooses through source_priority.
--
-- Build sequence (contract section 5.1)
--   Step 1  [0] settings and helpers  [1] ..__classified  [2] ..__inputs
--   Step 2  [3] area resolution for FHSIS  [4] part_* (one view per source)  [5] ..__candidate
--   Step 3  [6] ..__gates (severity 'fail' blocks, 'warn' is reported)  [7] build log  [8] STOP on failure
--   Step 4  [9] write  [10] build_log, build_inputs, row_disposition (rows not loaded only)  [11] summary
--
-- Measures are counts as published. Ratios in the sources (population per BHS etc.) are not
-- loaded: rates are computed in gold. '*' and '-' mean "no report" and become null; the
-- measure is listed in no_report_measures. Any other non-number fails a gate.
--
-- Needs: facilities_silver_00_reference.sql, quality_setup.sql, geography silver, bronze QC.
-- Status: ran on local Spark 4.0 against the landing files and the exported geography_silver
--   (2026-10-09). NOT tested locally: Delta, DESCRIBE HISTORY in a subquery ([2], [10]).
-- =============================================================================

-- [0] settings
DECLARE OR REPLACE VARIABLE metric_build_id STRING DEFAULT uuid();
DECLARE OR REPLACE VARIABLE silver_build_started_at TIMESTAMP DEFAULT current_timestamp();
DECLARE OR REPLACE VARIABLE strict_load_lookup BOOLEAN DEFAULT true;
DECLARE OR REPLACE VARIABLE code_version STRING DEFAULT NULL;
DECLARE OR REPLACE VARIABLE job_run_id STRING DEFAULT NULL;
DECLARE OR REPLACE VARIABLE geography_checksum STRING;
SET VARIABLE geography_checksum =
  (SELECT max_by(content_checksum, silver_processed_at) FROM public_health.silver.geography_silver);

-- Same helper functions as facilities_silver_01_registry.sql (kept identical in both files)
CREATE OR REPLACE TEMPORARY FUNCTION fac_norm(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(
         translate(upper(replace(replace(replace(replace(s, 'Ã±', 'N'), 'Ã‘', 'N'), 'ñ', 'N'), '?', 'N')), 'ÑÁÉÍÓÚ', 'NAEIOU'),
         '[^A-Z0-9]', ' '), ' +', ' '));

CREATE OR REPLACE TEMPORARY FUNCTION fac_place(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
         regexp_replace(regexp_replace(concat(' ', fac_norm(regexp_replace(s, "['’`-]", '')), ' '),
         ' (HUC|ICC|CC|CAPITAL|NOT A PROVINCE) ', ' '),
         ' STA ', ' SANTA '), ' STO ', ' SANTO '), ' GEN ', ' GENERAL '), ' GOV ', ' GOVERNOR '), ' SEN ', ' SENATOR '),
         '^ (MUNICIPALITY OF|ISLAND GARDEN CITY OF|SCIENCE CITY OF|CITY OF) ', ' '), ' CITY $', ' '), ' +', ' '));

CREATE OR REPLACE TEMPORARY FUNCTION fac_place_key(s STRING) RETURNS STRING
RETURN replace(fac_place(s), ' ', '');

CREATE OR REPLACE TEMPORARY FUNCTION fac_city_hint(s STRING) RETURNS BOOLEAN
RETURN fac_norm(s) RLIKE '(^CITY OF )|( CITY$)|( CITY )';

CREATE OR REPLACE TEMPORARY FUNCTION fac_num(s STRING) RETURNS DECIMAL(20,4)
RETURN try_cast(regexp_replace(trim(s), '[ ,]', '') AS DECIMAL(20,4));

-- 1 when a cell is neither empty, a number, nor a documented no-report marker
CREATE OR REPLACE TEMPORARY FUNCTION fac_parse_error(s STRING) RETURNS INT
RETURN CASE WHEN s IS NULL OR trim(s) IN ('', '*', '-', '#DIV/0!') THEN 0
            WHEN try_cast(regexp_replace(trim(s), '[ ,]', '') AS DECIMAL(20,4)) IS NOT NULL THEN 0
            ELSE 1 END;

CREATE OR REPLACE TEMPORARY FUNCTION fac_no_report(s STRING) RETURNS BOOLEAN
RETURN trim(s) IN ('*', '-');

CREATE OR REPLACE TEMPORARY FUNCTION fac_region_psgc(s STRING) RETURNS STRING
RETURN CASE
  WHEN fac_norm(s) RLIKE '(^| )(NCR|N C R|NATIONAL CAPITAL|METRO MANILA)( |$)'           THEN '1300000000'
  WHEN fac_norm(s) RLIKE '(^| )(CAR|C A R|CORDILLERA)( |$)'                               THEN '1400000000'
  WHEN fac_norm(s) RLIKE '(^| )(BARMM|ARMM|BANGSAMORO|AUTONOMOUS REGION IN MUSLIM MINDANAO)( |$)' THEN '1900000000'
  WHEN fac_norm(s) RLIKE '(^| )(NIR|NEGROS ISLAND)( |$)'                                  THEN '1800000000'
  WHEN fac_norm(s) RLIKE '(^| )(MIMAROPA|IV B|4 B|4B|IVB)( |$)'                           THEN '1700000000'
  WHEN fac_norm(s) RLIKE '(^| )(CARAGA|XIII|13)( |$)'                                     THEN '1600000000'
  WHEN fac_norm(s) RLIKE '(^| )(CALABARZON|IV A|4 A|4A|IVA)( |$)'                         THEN '0400000000'
  -- region names without a number (FHSIS 2019-2020: ILOCOS, CAGAYAN VALLEY, ...)
  WHEN fac_norm(s) RLIKE '^ILOCOS( REGION)?$'          THEN '0100000000'
  WHEN fac_norm(s) RLIKE '^CAGAYAN VALLEY( REGION)?$'  THEN '0200000000'
  WHEN fac_norm(s) RLIKE '^CENTRAL LUZON( REGION)?$'   THEN '0300000000'
  WHEN fac_norm(s) RLIKE '^BICOL( REGION)?$'           THEN '0500000000'
  WHEN fac_norm(s) RLIKE '^WESTERN VISAYAS( REGION)?$' THEN '0600000000'
  WHEN fac_norm(s) RLIKE '^CENTRAL VISAYAS( REGION)?$' THEN '0700000000'
  WHEN fac_norm(s) RLIKE '^EASTERN VISAYAS( REGION)?$' THEN '0800000000'
  WHEN fac_norm(s) RLIKE '^ZAMBOANGA PENINSULA( REGION)?$' THEN '0900000000'
  WHEN fac_norm(s) RLIKE '^NORTHERN MINDANAO( REGION)?$'   THEN '1000000000'
  WHEN fac_norm(s) RLIKE '^DAVAO( REGION)?$'           THEN '1100000000'
  WHEN fac_norm(s) RLIKE '^SOCCSKSARGEN( REGION)?$'    THEN '1200000000'
  ELSE CASE regexp_extract(fac_norm(s), '^REGION ([IVX]+|[0-9]+)( |$)', 1)
         WHEN 'I' THEN '0100000000' WHEN '1' THEN '0100000000' WHEN 'II' THEN '0200000000' WHEN '2' THEN '0200000000'
         WHEN 'III' THEN '0300000000' WHEN '3' THEN '0300000000' WHEN 'V' THEN '0500000000' WHEN '5' THEN '0500000000'
         WHEN 'VI' THEN '0600000000' WHEN '6' THEN '0600000000' WHEN 'VII' THEN '0700000000' WHEN '7' THEN '0700000000'
         WHEN 'VIII' THEN '0800000000' WHEN '8' THEN '0800000000' WHEN 'IX' THEN '0900000000' WHEN '9' THEN '0900000000'
         WHEN 'X' THEN '1000000000' WHEN '10' THEN '1000000000' WHEN 'XI' THEN '1100000000' WHEN '11' THEN '1100000000'
         WHEN 'XII' THEN '1200000000' WHEN '12' THEN '1200000000' END
  END;

-- Year at the end of a file name: ..._2024.xlsx -> 2024
CREATE OR REPLACE TEMPORARY FUNCTION fac_file_year(f STRING) RETURNS INT
RETURN try_cast(regexp_extract(f, '_([0-9]{4})[.][A-Za-z]+$', 1) AS INT);

CREATE OR REPLACE TEMP VIEW fac_geo AS
SELECT psgc_code, area_name,
       CASE geographic_level WHEN 'Reg' THEN 'region' WHEN 'Prov' THEN 'province' WHEN 'City' THEN 'city'
            WHEN 'Mun' THEN 'municipality' WHEN 'SubMun' THEN 'submunicipality' WHEN 'Bgy' THEN 'barangay' END AS geo_level,
       city_class, fac_place_key(area_name) AS place_norm,
       concat(substr(psgc_code, 1, 2), '00000000') AS region_psgc
FROM public_health.silver.geography_silver
WHERE content_checksum = geography_checksum;

-- Hospital counts before and after the 2013 definition change (plan 4.4)
CREATE OR REPLACE TEMPORARY FUNCTION fac_hosp_series(y INT) RETURNS STRING
RETURN CASE WHEN y <= 2012 THEN 'hospitals_incl_infirmaries' ELSE 'general_and_specialty_hospitals' END;


-- =============================================================================
-- [1] classify every bronze row
--   measures_per_row = silver rows one data row must produce (the unpivot rule, used in [6])
-- =============================================================================

-- PSA 9.18 hospitals and 9.17 beds, 2020-2024: 18 rows each, all data (national row + 17 regions)
CREATE OR REPLACE TEMP VIEW cls_psa_region_wide AS
SELECT 'public_health.bronze.facilities_psa_hospitals_region_annual_bronze' AS input_table,
       _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       Region AS label,
       CASE WHEN Region IS NULL THEN 'non_data' ELSE 'data' END AS row_class,
       CASE WHEN Region IS NULL THEN 'blank_label' END AS reason, 15 AS measures_per_row
FROM public_health.bronze.facilities_psa_hospitals_region_annual_bronze WHERE _source_key = 'psa_hospitals_region_annual'
UNION ALL
SELECT 'public_health.bronze.facilities_psa_beds_region_annual_bronze',
       _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       Regions, CASE WHEN Regions IS NULL THEN 'non_data' ELSE 'data' END,
       CASE WHEN Regions IS NULL THEN 'blank_label' END, 15
FROM public_health.bronze.facilities_psa_beds_region_annual_bronze WHERE _source_key = 'psa_beds_region_annual';

-- PSA OpenSTAT national hospitals and PSA 9.16 national hospitals + beds: a row is data when
-- its year cell is a 4-digit year; footnotes (a/ ... e/) and the source line are not.
CREATE OR REPLACE TEMP VIEW cls_psa_national AS
SELECT 'public_health.bronze.facilities_psa_hospitals_national_annual_bronze' AS input_table,
       _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       Year AS label,
       CASE WHEN trim(Year) RLIKE '^[0-9]{4}$' THEN 'data' ELSE 'non_data' END AS row_class,
       CASE WHEN NOT trim(Year) RLIKE '^[0-9]{4}$' THEN 'footnote_or_source' END AS reason, 3 AS measures_per_row
FROM public_health.bronze.facilities_psa_hospitals_national_annual_bronze WHERE _source_key = 'psa_hospitals_national_annual'
UNION ALL
SELECT 'public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze',
       _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       year, CASE WHEN trim(year) RLIKE '^[0-9]{4}$' THEN 'data' ELSE 'non_data' END,
       CASE WHEN NOT trim(year) RLIKE '^[0-9]{4}$' THEN 'footnote_or_source' END, 3
FROM public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze WHERE _source_key = 'psa_hospitals_beds_national_annual';

-- PSA 9.17 legacy (2005-2015): year blocks. A year row ('2005 ', '2013 a') opens a block;
-- the Total / Government / Private rows under it are data. The year is carried down.
CREATE OR REPLACE TEMP VIEW cls_psa_legacy AS
WITH r AS (
  SELECT t.*,
         try_cast(regexp_extract(trim(col_0), '^([0-9]{4})( a)?$', 1) AS INT) AS year_marker,
         (trim(col_0) RLIKE '^[0-9]{4} a$') AS year_footnote_a
  FROM public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze t
  WHERE _source_key = 'psa_hospitals_region_annual_legacy'
)
SELECT 'public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze' AS input_table, r.*,
       last_value(year_marker, true) OVER (PARTITION BY _source_file ORDER BY _source_row_num
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS block_year,
       last_value(CASE WHEN year_marker IS NOT NULL THEN year_footnote_a END, true) OVER (
         PARTITION BY _source_file ORDER BY _source_row_num ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS block_footnote_a,
       CASE WHEN trim(col_0) IN ('Total', 'Government', 'Private') THEN 'data' ELSE 'non_data' END AS row_class,
       CASE WHEN trim(col_0) IN ('Total', 'Government', 'Private') THEN NULL
            WHEN year_marker IS NOT NULL THEN 'year_header'
            WHEN _source_row_num <= 3 THEN 'title_or_header'
            WHEN upper(trim(col_0)) LIKE 'SOURCE%' THEN 'source_line'
            ELSE 'footnote' END AS reason,
       18 AS measures_per_row
FROM r;

-- PSA 9.19 BHS (2006-2023): a row is data when its year cell is a 4-digit year
CREATE OR REPLACE TEMP VIEW cls_psa_bhs AS
SELECT 'public_health.bronze.facilities_psa_bhs_region_annual_bronze' AS input_table, t.*,
       CASE WHEN trim(year) RLIKE '^[0-9]{4}$' THEN 'data' ELSE 'non_data' END AS row_class,
       CASE WHEN NOT trim(year) RLIKE '^[0-9]{4}$' THEN 'source_line' END AS reason,
       18 AS measures_per_row
FROM public_health.bronze.facilities_psa_bhs_region_annual_bronze t WHERE _source_key = 'psa_bhs_region_annual';

-- FHSIS BGY & BHS (2018-2025): data = a labelled row with at least one number or '*';
-- notes and legends (label, no values) and blank rows are not data.
CREATE OR REPLACE TEMP VIEW cls_fhsis AS
SELECT 'public_health.bronze.facilities_fhsis_primary_care_bronze' AS input_table, t.*,
       fac_file_year(_source_file) AS file_year,
       CASE WHEN area_name IS NOT NULL
                 AND coalesce(fac_num(population), fac_num(barangays_no), fac_num(bhs_no), fac_num(rhu_no),
                              fac_num(mhc_no), fac_num(chc_no), fac_num(health_centers_total)) IS NOT NULL
                 OR (area_name IS NOT NULL AND (fac_no_report(bhs_no) OR fac_no_report(barangays_no))) THEN 'data'
            WHEN area_name IS NOT NULL THEN 'non_data'
            WHEN coalesce(fac_num(population), fac_num(barangays_no), fac_num(bhs_no)) IS NULL THEN 'non_data'
            ELSE 'unclassified' END AS row_class,
       CASE WHEN area_name IS NOT NULL AND NOT (coalesce(fac_num(population), fac_num(barangays_no), fac_num(bhs_no),
                 fac_num(rhu_no), fac_num(mhc_no), fac_num(chc_no), fac_num(health_centers_total)) IS NOT NULL
                 OR fac_no_report(bhs_no) OR fac_no_report(barangays_no)) THEN 'note_or_legend'
            WHEN area_name IS NULL THEN 'blank_label' END AS reason,
       1 AS measures_per_row
FROM public_health.bronze.facilities_fhsis_primary_care_bronze t WHERE _source_key = 'doh_fhsis_demographic';

CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__classified AS
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       row_class, reason, measures_per_row FROM cls_psa_region_wide
UNION ALL
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       row_class, reason, measures_per_row FROM cls_psa_national
UNION ALL
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       row_class, reason, measures_per_row FROM cls_psa_legacy
UNION ALL
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       row_class, reason, measures_per_row FROM cls_psa_bhs
UNION ALL
SELECT input_table, _source_key, _source_file, _source_sheet, _source_row_num, _ingested_at, _as_of_date,
       row_class, reason, measures_per_row FROM cls_fhsis;


-- =============================================================================
-- [2] inputs
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_metric_declared_inputs AS
SELECT * FROM VALUES
  ('psa_hospitals_region_annual', 1), ('psa_beds_region_annual', 1), ('psa_hospitals_national_annual', 1),
  ('psa_hospitals_beds_national_annual', 1), ('psa_hospitals_region_annual_legacy', 1),
  ('psa_bhs_region_annual', 1), ('doh_fhsis_demographic', 8)
  AS t(source_key, expected_files);

CREATE OR REPLACE TEMP VIEW fac_metric_table_versions AS
SELECT 'public_health.bronze.facilities_psa_hospitals_region_annual_bronze' AS table_name,
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_hospitals_region_annual_bronze)) AS table_version
UNION ALL SELECT 'public_health.bronze.facilities_psa_beds_region_annual_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_beds_region_annual_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_psa_hospitals_national_annual_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_hospitals_national_annual_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_psa_bhs_region_annual_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_psa_bhs_region_annual_bronze))
UNION ALL SELECT 'public_health.bronze.facilities_fhsis_primary_care_bronze',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_fhsis_primary_care_bronze))
UNION ALL SELECT 'public_health.silver.geography_silver',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.geography_silver))
UNION ALL SELECT 'public_health.silver.facilities_area_exceptions',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_area_exceptions));

CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__inputs AS
WITH files AS (
  SELECT input_table, _source_key, _source_file, _source_sheet, _ingested_at, count(*) AS input_rows
  FROM public_health.silver.facilities_metric_silver__classified
  GROUP BY input_table, _source_key, _source_file, _source_sheet, _ingested_at
)
SELECT metric_build_id AS build_id, 'public_health.silver.facilities_metric_silver' AS target_table,
       'bronze_source' AS input_kind, f.input_table, max(v.table_version) AS input_table_version,
       f._source_key AS source_key, f._source_file AS source_file, f._source_sheet AS source_sheet,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.batch_id) END AS load_batch_id,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.content_checksum) END AS content_checksum,
       f._ingested_at AS ingested_at, f.input_rows, CAST(NULL AS STRING) AS reference_version,
       count(m.batch_id) AS metadata_matches
FROM files f
LEFT JOIN public_health.bronze.ingestion_metadata m
  ON m.source_name = f._source_key AND lower(m.status) = 'processed'
 AND endswith(m.raw_file_path, concat('/', f._source_file))
 AND m.source_sheet <=> f._source_sheet AND m.retrieved_at = f._ingested_at
LEFT JOIN fac_metric_table_versions v ON v.table_name = f.input_table
GROUP BY f.input_table, f._source_key, f._source_file, f._source_sheet, f._ingested_at, f.input_rows
UNION ALL
SELECT metric_build_id, 'public_health.silver.facilities_metric_silver', 'table', t.table_name, v.table_version,
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CASE WHEN t.table_name LIKE '%geography_silver' THEN geography_checksum END,
       CAST(NULL AS TIMESTAMP), t.n, CAST(NULL AS STRING), CAST(NULL AS BIGINT)
FROM (SELECT 'public_health.silver.geography_silver' AS table_name, count(*) AS n FROM fac_geo
      UNION ALL SELECT 'public_health.silver.facilities_area_exceptions', count(*) FROM public_health.silver.facilities_area_exceptions) t
LEFT JOIN fac_metric_table_versions v ON v.table_name = t.table_name;


-- =============================================================================
-- [3] FHSIS areas (plan 4.2 step 4): the national row, region rows (region rule), and the
--     province / city rows under each region (name + level + parent region). A province row
--     excludes the cities listed under it (plan 4.3).
--     Exceptions: source_group 'fhsis', match_type 'area_label', match_value = normalised
--     label, parent_label_norm = the parent REGION's PSGC code, optional year range.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__fhsis_geo AS
WITH r AS (
  SELECT c._source_key, c._source_file, c._source_sheet, c._source_row_num, c.file_year, c.area_name,
         fac_norm(c.area_name) AS label_norm, fac_place_key(c.area_name) AS place_key,
         fac_city_hint(c.area_name) AS city_hint,
         -- a region row: a region label that is not also a province name and does not say "city"
         -- ('CARAGA' is also a municipality and 'DAVAO' the place name of Davao City: still regions)
         CASE WHEN fac_norm(c.area_name) = 'PHILIPPINES' THEN 'national'
              WHEN fac_region_psgc(c.area_name) IS NOT NULL AND NOT fac_city_hint(c.area_name)
                   AND NOT EXISTS (SELECT 1 FROM fac_geo g WHERE g.geo_level = 'province'
                                     AND g.place_norm = fac_place_key(c.area_name)) THEN 'region'
              ELSE 'child' END AS row_kind
  FROM cls_fhsis c WHERE c.row_class = 'data'
),
rr AS (
  SELECT r.*,
         CASE WHEN row_kind = 'region' THEN fac_region_psgc(area_name) END AS own_region,
         last_value(CASE WHEN row_kind = 'region' THEN fac_region_psgc(area_name) END, true) OVER (
           PARTITION BY _source_file ORDER BY _source_row_num ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS parent_region,
         last_value(CASE WHEN row_kind = 'region' THEN trim(area_name) END, true) OVER (
           PARTITION BY _source_file ORDER BY _source_row_num ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS region_label
  FROM r
),
cand AS (
  SELECT rr._source_file, rr._source_row_num, g.psgc_code,
         CASE WHEN rr.city_hint AND g.geo_level = 'city' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level = 'province' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level IN ('city', 'municipality') THEN 1
              ELSE 0 END AS score
  FROM rr JOIN fac_geo g
    ON rr.row_kind = 'child' AND g.region_psgc = rr.parent_region AND g.place_norm = rr.place_key
   AND g.geo_level IN ('province', 'city', 'municipality')
   AND (NOT rr.city_hint OR g.geo_level = 'city')   -- a label that says city must be a city
),
-- No candidate in the published region: try the regions that took over part of it (Negros
-- Island Region 18 from VI and VII in 2024; BARMM 19 from XII), then the whole country.
-- A match there must still be unique.
related AS (
  SELECT * FROM VALUES ('0600000000', '1800000000'), ('0700000000', '1800000000'), ('1200000000', '1900000000'),
                       ('0900000000', '1900000000') AS t(published_region, successor_region)
),
cand_rel AS (
  SELECT rr._source_file, rr._source_row_num, g.psgc_code,
         CASE WHEN rr.city_hint AND g.geo_level = 'city' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level = 'province' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level IN ('city', 'municipality') THEN 1 ELSE 0 END AS score
  FROM rr
  JOIN related x ON x.published_region = rr.parent_region
  JOIN fac_geo g ON rr.row_kind = 'child' AND g.region_psgc = x.successor_region AND g.place_norm = rr.place_key
   AND g.geo_level IN ('province', 'city', 'municipality')
   AND (NOT rr.city_hint OR g.geo_level = 'city')   -- a label that says city must be a city
  WHERE NOT EXISTS (SELECT 1 FROM cand c WHERE c._source_file = rr._source_file AND c._source_row_num = rr._source_row_num)
),
cand_any AS (
  SELECT rr._source_file, rr._source_row_num, g.psgc_code,
         CASE WHEN rr.city_hint AND g.geo_level = 'city' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level = 'province' THEN 2
              WHEN NOT rr.city_hint AND g.geo_level IN ('city', 'municipality') THEN 1 ELSE 0 END AS score
  FROM rr
  JOIN fac_geo g ON rr.row_kind = 'child' AND g.place_norm = rr.place_key
   AND g.geo_level IN ('province', 'city', 'municipality')
   AND (NOT rr.city_hint OR g.geo_level = 'city')   -- a label that says city must be a city
  WHERE NOT EXISTS (SELECT 1 FROM cand c WHERE c._source_file = rr._source_file AND c._source_row_num = rr._source_row_num)
    AND NOT EXISTS (SELECT 1 FROM cand_rel c WHERE c._source_file = rr._source_file AND c._source_row_num = rr._source_row_num)
),
cand_all AS (
  SELECT *, 'name_level_parent' AS method FROM cand
  UNION ALL SELECT *, 'name_level_successor_region' FROM cand_rel
  UNION ALL SELECT *, 'name_level_other_region' FROM cand_any
),
best AS (
  SELECT _source_file, _source_row_num, count_if(score = max_s) AS n_best, max_by(psgc_code, score) AS best_psgc,
         max(method) AS method, array_join(array_sort(collect_list(psgc_code)), ',') AS candidates
  FROM (SELECT *, max(score) OVER (PARTITION BY _source_file, _source_row_num) AS max_s FROM cand_all)
  GROUP BY _source_file, _source_row_num
),
exc AS (SELECT * FROM public_health.silver.facilities_area_exceptions WHERE source_group = 'fhsis' AND match_type = 'area_label')
SELECT rr._source_key, rr._source_file, rr._source_sheet, rr._source_row_num, rr.row_kind, rr.region_label,
       CASE WHEN rr.row_kind = 'national' THEN 'ph'
            WHEN rr.row_kind = 'region' THEN concat('psgc:', rr.own_region)
            WHEN coalesce(e.psgc_code, e.area_key) IS NOT NULL THEN coalesce(concat('psgc:', e.psgc_code), e.area_key)
            WHEN b.n_best = 1 THEN concat('psgc:', b.best_psgc) END AS area_key,
       CASE WHEN rr.row_kind = 'region' THEN rr.own_region
            WHEN rr.row_kind = 'child' THEN coalesce(e.psgc_code, CASE WHEN e.area_key IS NULL AND b.n_best = 1 THEN b.best_psgc END) END AS psgc_code,
       CASE WHEN rr.row_kind = 'national' THEN 'national_row'
            WHEN rr.row_kind = 'region' THEN 'region_rule'
            WHEN coalesce(e.psgc_code, e.area_key) IS NOT NULL THEN 'exception'
            WHEN b.n_best = 1 THEN b.method END AS geo_resolution_method,
       b.candidates AS geo_candidates,
       e.reason AS geo_exception_reason
FROM rr
LEFT JOIN best b ON b._source_file = rr._source_file AND b._source_row_num = rr._source_row_num
LEFT JOIN exc e
  ON rr.row_kind = 'child' AND e.match_value = rr.label_norm
 AND (e.parent_label_norm IS NULL OR e.parent_label_norm = rr.parent_region)
 AND (e.year_from IS NULL OR rr.file_year >= CAST(e.year_from AS INT))
 AND (e.year_to IS NULL OR rr.file_year <= CAST(e.year_to AS INT));


-- =============================================================================
-- [4] one view per source, same columns in the same order. raw_* are the published cells;
--     [5] turns them into numbers. parse_errors counts cells that are not numbers or markers.
-- =============================================================================

-- [4a] PSA 9.18 hospitals and 9.17 beds: 15 columns (Total / Government / Private x 2020-2024)
CREATE OR REPLACE TEMP VIEW part_psa_region_wide
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
WITH b AS (
  SELECT 'hospitals' AS measure, Region AS label, Total_2020 AS t20, Total_2021 AS t21, Total_2022 AS t22, Total_2023 AS t23, Total_2024 AS t24,
         Government_2020 AS g20, Government_2021 AS g21, Government_2022 AS g22, Government_2023 AS g23, Government_2024 AS g24,
         Private_2020 AS p20, Private_2021 AS p21, Private_2022 AS p22, Private_2023 AS p23, Private_2024 AS p24,
         _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date
  FROM public_health.bronze.facilities_psa_hospitals_region_annual_bronze WHERE _source_key = 'psa_hospitals_region_annual' AND Region IS NOT NULL
  UNION ALL
  SELECT 'authorized_beds', Regions, Total_2020, Total_2021, Total_2022, Total_2023, Total_2024,
         Government_2020, Government_2021, Government_2022, Government_2023, Government_2024,
         Private_2020, Private_2021, Private_2022, Private_2023, Private_2024,
         _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date
  FROM public_health.bronze.facilities_psa_beds_region_annual_bronze WHERE _source_key = 'psa_beds_region_annual' AND Regions IS NOT NULL
),
u AS (
  SELECT b.*, stack(15,
     'total', 2020, t20, 'Total_2020', 'total', 2021, t21, 'Total_2021', 'total', 2022, t22, 'Total_2022',
     'total', 2023, t23, 'Total_2023', 'total', 2024, t24, 'Total_2024',
     'government', 2020, g20, 'Government_2020', 'government', 2021, g21, 'Government_2021', 'government', 2022, g22, 'Government_2022',
     'government', 2023, g23, 'Government_2023', 'government', 2024, g24, 'Government_2024',
     'private', 2020, p20, 'Private_2020', 'private', 2021, p21, 'Private_2021', 'private', 2022, p22, 'Private_2022',
     'private', 2023, p23, 'Private_2023', 'private', 2024, p24, 'Private_2024') AS (ownership, year, raw, src_col)
  FROM b
)
SELECT -- 9.17 labels its national BED row "Total Number of Hospitals" (source error, plan / YAML note)
       CASE WHEN label LIKE 'Total Number of Hospitals%' THEN 'ph' ELSE concat('psgc:', fac_region_psgc(label)) END AS area_key,
       CASE WHEN label LIKE 'Total Number of Hospitals%' THEN NULL ELSE fac_region_psgc(label) END AS psgc_code,
       trim(label) AS geo_name_source,
       CASE WHEN label LIKE 'Total Number of Hospitals%' THEN 'national_row' ELSE 'region_rule' END AS geo_resolution_method,
       CASE WHEN label LIKE 'Total Number of Hospitals%' THEN NULL ELSE trim(label) END AS region_as_published,
       year, ownership,
       CASE WHEN measure = 'hospitals' THEN raw END AS raw_hospitals,
       CASE WHEN measure = 'authorized_beds' THEN raw END AS raw_authorized_beds,
       CAST(NULL AS STRING) AS raw_bhs, CAST(NULL AS STRING) AS raw_rhu, CAST(NULL AS STRING) AS raw_mhc,
       CAST(NULL AS STRING) AS raw_chc, CAST(NULL AS STRING) AS raw_health_centers_total,
       CAST(NULL AS STRING) AS raw_barangays, CAST(NULL AS STRING) AS raw_population,
       CASE WHEN measure = 'hospitals' THEN fac_hosp_series(year) END AS series_definition,
       false AS is_break_year,
       'regions_17_old_definition' AS region_definition,
       _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date,
       map(measure, src_col) AS _metric_source_columns
FROM u;

-- [4b] PSA OpenSTAT national hospitals and PSA 9.16 national hospitals + beds
CREATE OR REPLACE TEMP VIEW part_psa_national
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
WITH b AS (
  SELECT CAST(trim(Year) AS INT) AS y, Total AS h_t, Government AS h_g, Private AS h_p,
         CAST(NULL AS STRING) AS b_t, CAST(NULL AS STRING) AS b_g, CAST(NULL AS STRING) AS b_p,
         'Total' AS hc_t, 'Government' AS hc_g, 'Private' AS hc_p, CAST(NULL AS STRING) AS bc_t, CAST(NULL AS STRING) AS bc_g, CAST(NULL AS STRING) AS bc_p,
         _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date
  FROM public_health.bronze.facilities_psa_hospitals_national_annual_bronze
  WHERE _source_key = 'psa_hospitals_national_annual' AND trim(Year) RLIKE '^[0-9]{4}$'
  UNION ALL
  SELECT CAST(trim(year) AS INT), hospitals_total, hospitals_government, hospitals_private,
         beds_total, beds_government, beds_private,
         'hospitals_total', 'hospitals_government', 'hospitals_private', 'beds_total', 'beds_government', 'beds_private',
         _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date
  FROM public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze
  WHERE _source_key = 'psa_hospitals_beds_national_annual' AND trim(year) RLIKE '^[0-9]{4}$'
),
u AS (
  SELECT b.*, stack(3, 'total', h_t, b_t, hc_t, bc_t, 'government', h_g, b_g, hc_g, bc_g, 'private', h_p, b_p, hc_p, bc_p)
         AS (ownership, raw_h, raw_b, col_h, col_b)
  FROM b
)
SELECT 'ph', CAST(NULL AS STRING), 'Philippines', 'national_row', CAST(NULL AS STRING),
       y, ownership, raw_h, raw_b,
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS STRING), CAST(NULL AS STRING),
       fac_hosp_series(y), (y = 2013), 'national',
       _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date,
       CASE WHEN col_b IS NULL THEN map('hospitals', col_h) ELSE map('hospitals', col_h, 'authorized_beds', col_b) END
FROM u;

-- [4c] PSA 9.17 legacy: one data row = one ownership in one year, 18 areas (Philippines + 17 regions).
--      The column labelled BARMM holds ARMM for these years.
CREATE OR REPLACE TEMP VIEW part_psa_legacy
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
WITH u AS (
  SELECT c.*, stack(18,
     'Philippines', col_1, 'col_1', 'NCR', col_3, 'col_3', 'CAR', col_4, 'col_4', 'Region I', col_5, 'col_5',
     'Region II', col_6, 'col_6', 'Region III', col_7, 'col_7', 'Region IV-A', col_8, 'col_8', 'MIMAROPA', col_9, 'col_9',
     'Region V', col_10, 'col_10', 'Region VI', col_11, 'col_11', 'Region VII', col_12, 'col_12',
     'Region VIII', col_13, 'col_13', 'Region IX', col_14, 'col_14', 'Region X', col_15, 'col_15',
     'Region XI', col_16, 'col_16', 'Region XII', col_17, 'col_17', 'Caraga', col_18, 'col_18', 'ARMM', col_19, 'col_19'
  ) AS (area_label, raw, src_col)
  FROM cls_psa_legacy c WHERE c.row_class = 'data'
)
SELECT CASE WHEN area_label = 'Philippines' THEN 'ph' ELSE concat('psgc:', fac_region_psgc(area_label)) END,
       CASE WHEN area_label = 'Philippines' THEN NULL ELSE fac_region_psgc(area_label) END,
       area_label,
       CASE WHEN area_label = 'Philippines' THEN 'national_row' ELSE 'column_position' END,
       CASE WHEN area_label = 'Philippines' THEN NULL ELSE area_label END,
       block_year, lower(trim(col_0)), raw, CAST(NULL AS STRING),
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS STRING), CAST(NULL AS STRING),
       fac_hosp_series(block_year), (block_year = 2013),
       CASE WHEN area_label = 'Philippines' THEN 'national' ELSE 'regions_17_old_definition' END,
       _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date,
       map('hospitals', src_col)
FROM u;

-- [4d] PSA 9.19 BHS: one data row = one year, 18 areas
CREATE OR REPLACE TEMP VIEW part_psa_bhs
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
WITH u AS (
  SELECT c.*, stack(18,
     'Philippines', philippines, 'philippines', 'NCR', ncr, 'ncr', 'CAR', car, 'car', 'Region I', region_01, 'region_01',
     'Region II', region_02, 'region_02', 'Region III', region_03, 'region_03', 'Region IV-A', region_04a, 'region_04a',
     'MIMAROPA', mimaropa, 'mimaropa', 'Region V', region_05, 'region_05', 'Region VI', region_06, 'region_06',
     'Region VII', region_07, 'region_07', 'Region VIII', region_08, 'region_08', 'Region IX', region_09, 'region_09',
     'Region X', region_10, 'region_10', 'Region XI', region_11, 'region_11', 'Region XII', region_12, 'region_12',
     'Caraga', caraga, 'caraga', 'BARMM', barmm, 'barmm'
  ) AS (area_label, raw, src_col)
  FROM cls_psa_bhs c WHERE c.row_class = 'data'
)
SELECT CASE WHEN area_label = 'Philippines' THEN 'ph' ELSE concat('psgc:', fac_region_psgc(area_label)) END,
       CASE WHEN area_label = 'Philippines' THEN NULL ELSE fac_region_psgc(area_label) END,
       area_label,
       CASE WHEN area_label = 'Philippines' THEN 'national_row' ELSE 'column_position' END,
       CASE WHEN area_label = 'Philippines' THEN NULL
            WHEN area_label = 'BARMM' AND CAST(trim(year) AS INT) < 2019 THEN 'ARMM' ELSE area_label END,
       CAST(trim(year) AS INT), 'total',
       CAST(NULL AS STRING), CAST(NULL AS STRING), raw,
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS STRING), false,
       CASE WHEN area_label = 'Philippines' THEN 'national' ELSE 'regions_17_old_definition' END,
       _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date,
       map('bhs', src_col)
FROM u;

-- [4e] FHSIS BGY & BHS. Columns change by year (plan 4.4):
--   2018       barangays, BHS
--   2019-2024  barangays, MHC, CHC, RHU, health centres total, BHS
--   2025       barangays, "RHU" (= all health centres, decision D4), BHS
CREATE OR REPLACE TEMP VIEW part_fhsis
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
SELECT g.area_key, g.psgc_code, trim(c.area_name),
       g.geo_resolution_method,
       g.region_label,
       c.file_year, 'total',
       CAST(NULL AS STRING), CAST(NULL AS STRING), c.bhs_no,
       CASE WHEN c.file_year BETWEEN 2019 AND 2024 THEN c.rhu_no END,
       c.mhc_no, c.chc_no,
       CASE WHEN c.file_year = 2025 THEN c.rhu_no ELSE c.health_centers_total END,
       c.barangays_no, c.population,
       CAST(NULL AS STRING), false,
       CASE WHEN g.row_kind = 'national' THEN 'national' WHEN g.row_kind = 'region' THEN 'region_as_reported'
            ELSE 'province_excl_listed_cities_or_city' END,
       c._source_key, c._source_file, c._source_sheet, c._source_row_num, c._as_of_date,
       map_filter(map(
         'bhs', 'bhs_no', 'barangays', 'barangays_no', 'population', 'population',
         'rhu', CASE WHEN c.file_year BETWEEN 2019 AND 2024 THEN 'rhu_no' END,
         'mhc', CASE WHEN c.file_year BETWEEN 2019 AND 2024 THEN 'mhc_no' END,
         'chc', CASE WHEN c.file_year BETWEEN 2019 AND 2024 THEN 'chc_no' END,
         'health_centers_total', CASE WHEN c.file_year = 2025 THEN 'rhu_no'
                                      WHEN c.file_year BETWEEN 2019 AND 2024 THEN 'health_centers_total' END),
         (k, v) -> v IS NOT NULL)
FROM cls_fhsis c
JOIN public_health.silver.facilities_metric_silver__fhsis_geo g
  ON g._source_file = c._source_file AND g._source_sheet <=> c._source_sheet AND g._source_row_num = c._source_row_num
WHERE c.row_class = 'data';


-- =============================================================================
-- [5] candidate rows
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_metric_parts
  (area_key, psgc_code, geo_name_source, geo_resolution_method, region_as_published, year, ownership,
   raw_hospitals, raw_authorized_beds, raw_bhs, raw_rhu, raw_mhc, raw_chc, raw_health_centers_total,
   raw_barangays, raw_population, series_definition, is_break_year, geo_coverage_basis,
   _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns) AS
SELECT * FROM part_psa_region_wide
UNION ALL SELECT * FROM part_psa_national
UNION ALL SELECT * FROM part_psa_legacy
UNION ALL SELECT * FROM part_psa_bhs
UNION ALL SELECT * FROM part_fhsis;

CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__candidate AS
SELECT area_key, psgc_code, geo_name_source, geo_resolution_method,
       -- 'mislabelled_row' marks a proposed correction (plan: excluded from area rankings until reviewed)
       geo_exception_reason,
       region_as_published, geo_coverage_basis,
       year, ownership,
       CAST(fac_num(raw_hospitals) AS INT)            AS hospitals,
       CAST(fac_num(raw_authorized_beds) AS INT)      AS authorized_beds,
       CAST(fac_num(raw_bhs) AS INT)                  AS bhs,
       CAST(fac_num(raw_rhu) AS INT)                  AS rhu,
       CAST(fac_num(raw_mhc) AS INT)                  AS mhc,
       CAST(fac_num(raw_chc) AS INT)                  AS chc,
       CAST(fac_num(raw_health_centers_total) AS INT) AS health_centers_total,
       CAST(fac_num(raw_barangays) AS INT)            AS barangays,
       CAST(round(fac_num(raw_population)) AS BIGINT) AS population,   -- some FHSIS years carry decimals
       filter(array(
         CASE WHEN fac_no_report(raw_hospitals) THEN 'hospitals' END,
         CASE WHEN fac_no_report(raw_authorized_beds) THEN 'authorized_beds' END,
         CASE WHEN fac_no_report(raw_bhs) THEN 'bhs' END, CASE WHEN fac_no_report(raw_rhu) THEN 'rhu' END,
         CASE WHEN fac_no_report(raw_mhc) THEN 'mhc' END, CASE WHEN fac_no_report(raw_chc) THEN 'chc' END,
         CASE WHEN fac_no_report(raw_health_centers_total) THEN 'health_centers_total' END,
         CASE WHEN fac_no_report(raw_barangays) THEN 'barangays' END,
         CASE WHEN fac_no_report(raw_population) THEN 'population' END), x -> x IS NOT NULL) AS no_report_measures,
       series_definition, is_break_year,
       metric_build_id AS _silver_build_id,
       _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns,
       -- validation only, not published
       fac_parse_error(raw_hospitals) + fac_parse_error(raw_authorized_beds) + fac_parse_error(raw_bhs)
       + fac_parse_error(raw_rhu) + fac_parse_error(raw_mhc) + fac_parse_error(raw_chc)
       + fac_parse_error(raw_health_centers_total) + fac_parse_error(raw_barangays)
       + fac_parse_error(raw_population) AS parse_errors
FROM fac_metric_parts p
LEFT JOIN (SELECT _source_file AS g_file, _source_row_num AS g_row, geo_exception_reason
           FROM public_health.silver.facilities_metric_silver__fhsis_geo) g
  ON p._source_key = 'doh_fhsis_demographic' AND g.g_file = p._source_file AND g.g_row = p._source_row_num;


-- =============================================================================
-- [5b] dispositions (lineage-qc-contract.md v1.3, section 7.1): one row per bronze row that was
--      NOT loaded. Loaded rows are not stored: the silver rows carry their bronze row reference, and
--      rows loaded per input = input_rows - rows here (quality.v_build_reconciliation).
--      Non-data rows (titles, footnotes, spacers, year headers) are the only rows not loaded.
--      Saved before the gates, so the gates check exactly what step [10] writes.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__dispositions AS
SELECT metric_build_id AS build_id, 'public_health.silver.facilities_metric_silver' AS target_table,
       c._source_key AS source_key, c._source_file AS source_file, c._source_sheet AS source_sheet,
       c._source_row_num AS source_row_num, i.load_batch_id, i.content_checksum,
       'dropped_non_data' AS disposition, c.reason, CAST(NULL AS STRING) AS target_key,
       current_timestamp() AS recorded_at
FROM public_health.silver.facilities_metric_silver__classified c
LEFT JOIN public_health.silver.facilities_metric_silver__inputs i
  ON i.input_kind = 'bronze_source' AND i.source_key = c._source_key AND i.source_file = c._source_file
 AND i.source_sheet <=> c._source_sheet
WHERE c.row_class <> 'data';


-- =============================================================================
-- [6] gates. severity 'fail' stops the build; 'warn' is reported in the summary only.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver__gates AS
SELECT 'inputs: declared source missing or with a different number of files' AS gate, 'fail' AS severity, count(*) AS failures
  FROM fac_metric_declared_inputs d
  LEFT JOIN (SELECT source_key, count(*) AS n FROM public_health.silver.facilities_metric_silver__inputs
             WHERE input_kind = 'bronze_source' GROUP BY source_key) i ON i.source_key = d.source_key
  WHERE coalesce(i.n, 0) <> d.expected_files
UNION ALL
SELECT 'inputs: file or sheet with more than one load time', 'fail', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_metric_silver__inputs WHERE input_kind = 'bronze_source'
  GROUP BY input_table, source_key, source_file, source_sheet HAVING count(*) > 1)
UNION ALL
SELECT 'inputs: more than one ingestion log match', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__inputs WHERE input_kind = 'bronze_source' AND metadata_matches > 1
UNION ALL
SELECT 'inputs: no processed ingestion log match (strict_load_lookup = true)', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__inputs
  WHERE input_kind = 'bronze_source' AND metadata_matches = 0 AND strict_load_lookup
UNION ALL
SELECT 'unclassified bronze rows', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__classified WHERE row_class NOT IN ('data', 'non_data')
UNION ALL
SELECT 'bronze row classified more than once', 'fail', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_metric_silver__classified
  GROUP BY _source_key, _source_file, _source_sheet, _source_row_num HAVING count(*) > 1)
UNION ALL
SELECT 'PSA 9.17 legacy: region header row is not NCR ... BARMM in columns 3 to 19', 'fail',
       CASE WHEN EXISTS (SELECT 1 FROM public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze
                         WHERE _source_key = 'psa_hospitals_region_annual_legacy'
                           AND trim(col_3) = 'NCR' AND trim(col_4) = 'CAR' AND trim(col_5) = 'I' AND trim(col_8) = 'IV-A'
                           AND trim(col_9) = 'MIMAROPA' AND trim(col_17) = 'XII' AND trim(col_18) = 'Caraga'
                           AND trim(col_19) = 'BARMM') THEN 0 ELSE 1 END
UNION ALL
SELECT 'PSA 9.17 legacy: data row without a year above it', 'fail', count(*)
  FROM cls_psa_legacy WHERE row_class = 'data' AND block_year IS NULL
UNION ALL
SELECT 'PSA 9.19 BHS: header row is not Year / Philippines / Region', 'fail', 0   -- checked by bronze header_check
UNION ALL
SELECT 'region label not recognised', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__candidate WHERE area_key IS NULL OR area_key = 'psgc:'
UNION ALL
SELECT 'FHSIS: area not resolved (add an area exception with the parent region code)', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__fhsis_geo WHERE area_key IS NULL
UNION ALL
SELECT 'area code not in geography', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__candidate c
  WHERE c.psgc_code IS NOT NULL AND NOT EXISTS (SELECT 1 FROM fac_geo g WHERE g.psgc_code = c.psgc_code)
UNION ALL
SELECT 'cell that is neither a number nor a documented marker', 'fail', coalesce(sum(parse_errors), 0)
  FROM public_health.silver.facilities_metric_silver__candidate
UNION ALL
SELECT 'negative count', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__candidate
  WHERE least(hospitals, authorized_beds, bhs, rhu, mhc, chc, health_centers_total, barangays, population) < 0
UNION ALL
SELECT 'key column null', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__candidate
  WHERE area_key IS NULL OR year IS NULL OR ownership IS NULL OR _source_key IS NULL
UNION ALL
SELECT 'key not unique', 'fail', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_metric_silver__candidate
  GROUP BY area_key, year, ownership, _source_key HAVING count(*) > 1)
UNION ALL
SELECT 'lineage: a lineage column is null (only _source_sheet may be)', 'fail', count(*)
  FROM public_health.silver.facilities_metric_silver__candidate
  WHERE _silver_build_id IS NULL OR _source_key IS NULL OR _source_file IS NULL OR _source_row_num IS NULL
     OR _as_of_date IS NULL OR _metric_source_columns IS NULL OR size(_metric_source_columns) = 0
UNION ALL
-- contract 7.1, per input: distinct bronze rows referenced by silver = rows read - rows not loaded
SELECT 'reconciliation: inputs where rows referenced by silver <> rows read - rows not loaded', 'fail', count(*)
FROM public_health.silver.facilities_metric_silver__inputs i
LEFT JOIN (
  SELECT _source_key, _source_file, _source_sheet, count(*) AS n FROM (
    SELECT DISTINCT _source_key, _source_file, _source_sheet, _source_row_num FROM public_health.silver.facilities_metric_silver__candidate)
  GROUP BY _source_key, _source_file, _source_sheet) ref
  ON ref._source_key = i.source_key AND ref._source_file = i.source_file AND ref._source_sheet <=> i.source_sheet
LEFT JOIN (
  SELECT source_key, source_file, source_sheet, count(*) AS n
  FROM public_health.silver.facilities_metric_silver__dispositions GROUP BY source_key, source_file, source_sheet) nl
  ON nl.source_key = i.source_key AND nl.source_file = i.source_file AND nl.source_sheet <=> i.source_sheet
WHERE i.input_kind = 'bronze_source' AND i.input_rows - coalesce(nl.n, 0) <> coalesce(ref.n, 0)
UNION ALL
-- output validation per source
SELECT concat(c.source_key, ': silver rows <> data rows x measures per row'), 'fail',
       abs(c.expected_rows - coalesce(s.silver_rows, 0))
  FROM (SELECT _source_key AS source_key, sum(CASE WHEN row_class = 'data' THEN measures_per_row ELSE 0 END) AS expected_rows
        FROM public_health.silver.facilities_metric_silver__classified GROUP BY _source_key) c
  LEFT JOIN (SELECT _source_key AS source_key, count(*) AS silver_rows
             FROM public_health.silver.facilities_metric_silver__candidate GROUP BY _source_key) s ON s.source_key = c.source_key
UNION ALL
SELECT concat(c.source_key, ': bronze rows referenced by silver lineage <> data rows'), 'fail',
       abs(c.data_rows - coalesce(s.referenced, 0))
  FROM (SELECT _source_key AS source_key, count_if(row_class = 'data') AS data_rows
        FROM public_health.silver.facilities_metric_silver__classified GROUP BY _source_key) c
  LEFT JOIN (SELECT _source_key AS source_key, count(*) AS referenced
             FROM (SELECT DISTINCT _source_key, _source_file, _source_sheet, _source_row_num FROM public_health.silver.facilities_metric_silver__candidate)
             GROUP BY _source_key) s ON s.source_key = c.source_key
UNION ALL
-- control values (the same numbers bronze QC uses)
SELECT concat('control: ', t.label), 'fail', CASE WHEN a.v = t.expected THEN 0 ELSE 1 END
FROM (SELECT * FROM VALUES
  ('PSA 9.18 hospitals 2024 = 1,312', 'psa_hospitals_region_annual', 2024, 'hospitals', 1312),
  ('PSA 9.18 hospitals 2020 = 1,283', 'psa_hospitals_region_annual', 2020, 'hospitals', 1283),
  ('PSA 9.17 beds 2024 = 114,623', 'psa_beds_region_annual', 2024, 'authorized_beds', 114623),
  ('PSA 9.17 beds 2020 = 110,136', 'psa_beds_region_annual', 2020, 'authorized_beds', 110136),
  ('OpenSTAT hospitals 2006 = 1,921', 'psa_hospitals_national_annual', 2006, 'hospitals', 1921),
  ('PSA 9.16 beds 1990 = 87,133', 'psa_hospitals_beds_national_annual', 1990, 'authorized_beds', 87133),
  ('PSA 9.16 beds 2014 = 98,429', 'psa_hospitals_beds_national_annual', 2014, 'authorized_beds', 98429),
  ('PSA 9.17 legacy hospitals 2005 = 1,838', 'psa_hospitals_region_annual_legacy', 2005, 'hospitals', 1838),
  ('PSA 9.19 BHS 2023 = 24,327', 'psa_bhs_region_annual', 2023, 'bhs', 24327),
  ('FHSIS BHS 2018 = 21,546', 'doh_fhsis_demographic', 2018, 'bhs', 21546),
  ('FHSIS BHS 2024 = 25,532', 'doh_fhsis_demographic', 2024, 'bhs', 25532),
  ('FHSIS BHS 2025 = 26,944', 'doh_fhsis_demographic', 2025, 'bhs', 26944),
  ('FHSIS barangays 2024 = 41,590', 'doh_fhsis_demographic', 2024, 'barangays', 41590)
  AS v(label, k, y, measure, expected)) t
LEFT JOIN (SELECT _source_key, year, measure, v
           FROM (SELECT _source_key, year, stack(4, 'hospitals', CAST(hospitals AS BIGINT), 'authorized_beds', CAST(authorized_beds AS BIGINT),
                                                    'bhs', CAST(bhs AS BIGINT), 'barangays', CAST(barangays AS BIGINT)) AS (measure, v)
                 FROM public_health.silver.facilities_metric_silver__candidate
                 WHERE area_key = 'ph' AND ownership = 'total')) a
  ON a._source_key = t.k AND a.year = t.y AND a.measure = t.measure
UNION ALL
-- consistency (reported, not blocking): a region equals its provinces plus its listed cities
SELECT 'warn: FHSIS region BHS <> sum of the rows listed under it (region-years)', 'warn', count(*) FROM (
  SELECT r.year, r.area_key FROM public_health.silver.facilities_metric_silver__candidate r
  JOIN public_health.silver.facilities_metric_silver__fhsis_geo g
    ON g._source_file = r._source_file AND g._source_row_num = r._source_row_num AND g.row_kind = 'region'
  JOIN (SELECT c.year, g2.region_label, c._source_file, sum(c.bhs) AS child_bhs
        FROM public_health.silver.facilities_metric_silver__candidate c
        JOIN public_health.silver.facilities_metric_silver__fhsis_geo g2
          ON g2._source_file = c._source_file AND g2._source_row_num = c._source_row_num AND g2.row_kind = 'child'
        GROUP BY c.year, g2.region_label, c._source_file) ch
    ON ch._source_file = r._source_file AND ch.region_label = r.geo_name_source
  WHERE r._source_key = 'doh_fhsis_demographic' AND NOT (r.bhs <=> ch.child_bhs))
UNION ALL
SELECT 'warn: PSA 9.18 / 9.17 regions do not add up to the national row (area-years)', 'warn', count(*) FROM (
  SELECT c._source_key, c.year, c.ownership
  FROM public_health.silver.facilities_metric_silver__candidate c
  WHERE c._source_key IN ('psa_hospitals_region_annual', 'psa_beds_region_annual')
  GROUP BY c._source_key, c.year, c.ownership
  HAVING sum(CASE WHEN area_key <> 'ph' THEN coalesce(hospitals, authorized_beds) END)
         <> sum(CASE WHEN area_key = 'ph' THEN coalesce(hospitals, authorized_beds) END));


-- =============================================================================
-- [7] build log row
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_metric_build_log AS
SELECT metric_build_id AS build_id, 'silver' AS layer, 'facilities' AS domain,
       'public_health.silver.facilities_metric_silver' AS target_table,
       silver_build_started_at AS started_at, current_timestamp() AS finished_at,
       CASE WHEN g.failed_gates = 0 THEN 'succeeded' ELSE 'failed' END AS status,
       CASE WHEN g.failed_gates = 0 THEN (SELECT count(*) FROM public_health.silver.facilities_metric_silver__candidate) END AS row_count,
       CAST(NULL AS BIGINT) AS target_table_version, code_version AS code_version,
       to_json(named_struct('strict_load_lookup', strict_load_lookup, 'geography_checksum', geography_checksum,
                            'warnings', g.warn_names)) AS parameters,
       job_run_id AS run_id,
       CASE WHEN g.failed_gates > 0 THEN substr(concat('gates failed: ', g.failed_names), 1, 500) END AS error_message
FROM (SELECT count_if(failures <> 0 AND severity = 'fail') AS failed_gates,
             concat_ws(' | ', collect_list(CASE WHEN failures <> 0 AND severity = 'fail' THEN concat(gate, ' = ', failures) END)) AS failed_names,
             concat_ws(' | ', collect_list(CASE WHEN failures <> 0 AND severity = 'warn' THEN concat(gate, ' = ', failures) END)) AS warn_names
      FROM public_health.silver.facilities_metric_silver__gates) g;

-- [8] stop if a gate failed (the failed build is logged first)
INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
       target_table_version, code_version, parameters, run_id, error_message
FROM fac_metric_build_log WHERE status = 'failed';

SELECT raise_error(concat('facilities_metric_silver was NOT published. ', error_message))
FROM fac_metric_build_log WHERE status = 'failed';


-- =============================================================================
-- [9] write (full rebuild)
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_metric_silver AS
SELECT * EXCEPT (parse_errors) FROM public_health.silver.facilities_metric_silver__candidate;


-- =============================================================================
-- [10] record the build
-- =============================================================================
INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status,
       (SELECT count(*) FROM public_health.silver.facilities_metric_silver),
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_metric_silver)),
       code_version, parameters, run_id, error_message
FROM fac_metric_build_log;

INSERT INTO public_health.quality.build_inputs
  (build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
   source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version)
SELECT build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
       source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version
FROM public_health.silver.facilities_metric_silver__inputs;

-- bronze rows not loaded (staged in [5b]); loaded rows are not written (contract 7.1)
INSERT INTO public_health.quality.row_disposition
  (build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
   content_checksum, disposition, reason, target_key, recorded_at)
SELECT build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
       content_checksum, disposition, reason, target_key, recorded_at
FROM public_health.silver.facilities_metric_silver__dispositions;


-- =============================================================================
-- [11] summary and warnings; drop staging tables (__classified kept until the next run)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_metric_summary AS
SELECT _source_key, min(year) AS year_from, max(year) AS year_to, count(*) AS silver_rows,
       count(DISTINCT area_key) AS areas, sum(size(no_report_measures)) AS no_report_cells
FROM public_health.silver.facilities_metric_silver GROUP BY _source_key;

CREATE OR REPLACE TEMP VIEW fac_metric_warnings AS
SELECT gate, failures FROM public_health.silver.facilities_metric_silver__gates WHERE severity = 'warn';

SELECT * FROM fac_metric_warnings;

DROP TABLE IF EXISTS public_health.silver.facilities_metric_silver__candidate;
DROP TABLE IF EXISTS public_health.silver.facilities_metric_silver__gates;
DROP TABLE IF EXISTS public_health.silver.facilities_metric_silver__inputs;
DROP TABLE IF EXISTS public_health.silver.facilities_metric_silver__fhsis_geo;
DROP TABLE IF EXISTS public_health.silver.facilities_metric_silver__dispositions;

SELECT * FROM fac_metric_summary ORDER BY _source_key;

In [0]:
-- =============================================================================
-- facilities_silver_03_bed_occupancy.sql
-- Builds public_health.silver.facilities_bed_occupancy_silver:
--   one anonymous hospital record (no name, no code) of one year from the bed occupancy file.
--   Key: (_source_key, _source_file, _source_sheet, _source_row_num). The records are
--   anonymous, so the lineage columns are the key; identical rows are kept apart (plan 3, 5.2).
-- Follows facilities-silver-plan.md (section 3: occupancy definitions) and lineage-qc-contract.md v1.2.
--
-- SHARING: restricted. The file's release terms are not confirmed (plan 9). Keep the table's
-- access limited and publish only aggregates with small cells suppressed.
--
-- What the data is (plan 3.1)
--   2020-2024 sheets: PRIVATE hospitals only, city/municipality level, BOR only.
--   2025 sheet: government and private, province level, BOR + authorized and implementing beds.
--   The 2025 records are the preferred CANDIDATE occupancy baseline, subject to validation of
--   bed definitions, duplicate records and reporting coverage (decision D6).
--
-- Build sequence: [0] settings  [1] classify  [2] inputs  [3] areas  [4] candidate
--                 [4b] dispositions  [5] gates  [6] build log  [7] STOP on failure  [8] write  [9] record  [10] summary
-- Needs: facilities_silver_00_reference.sql, quality_setup.sql, geography silver, bronze QC.
-- Status: ran on local Spark 4.0 against the landing file and the exported geography_silver
--   (2026-10-09). NOT tested locally: Delta, DESCRIBE HISTORY in a subquery ([2], [9]).
-- =============================================================================

-- [0] settings
DECLARE OR REPLACE VARIABLE bor_build_id STRING DEFAULT uuid();
DECLARE OR REPLACE VARIABLE silver_build_started_at TIMESTAMP DEFAULT current_timestamp();
DECLARE OR REPLACE VARIABLE strict_load_lookup BOOLEAN DEFAULT true;
DECLARE OR REPLACE VARIABLE code_version STRING DEFAULT NULL;
DECLARE OR REPLACE VARIABLE job_run_id STRING DEFAULT NULL;
-- BOR above this value is kept and flagged but is not eligible for the baseline (plan 3.3)
DECLARE OR REPLACE VARIABLE bor_baseline_max DECIMAL(7,2) DEFAULT 150;
DECLARE OR REPLACE VARIABLE geography_checksum STRING;
SET VARIABLE geography_checksum =
  (SELECT max_by(content_checksum, silver_processed_at) FROM public_health.silver.geography_silver);

-- Same helper functions as facilities_silver_01_registry.sql (kept identical)
CREATE OR REPLACE TEMPORARY FUNCTION fac_norm(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(
         translate(upper(replace(replace(replace(replace(s, 'Ã±', 'N'), 'Ã‘', 'N'), 'ñ', 'N'), '?', 'N')), 'ÑÁÉÍÓÚ', 'NAEIOU'),
         '[^A-Z0-9]', ' '), ' +', ' '));

CREATE OR REPLACE TEMPORARY FUNCTION fac_place(s STRING) RETURNS STRING
RETURN trim(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
         regexp_replace(regexp_replace(concat(' ', fac_norm(regexp_replace(s, "['’`-]", '')), ' '),
         ' (HUC|ICC|CC|CAPITAL|NOT A PROVINCE) ', ' '),
         ' STA ', ' SANTA '), ' STO ', ' SANTO '), ' GEN ', ' GENERAL '), ' GOV ', ' GOVERNOR '), ' SEN ', ' SENATOR '),
         '^ (MUNICIPALITY OF|ISLAND GARDEN CITY OF|SCIENCE CITY OF|CITY OF) ', ' '), ' CITY $', ' '), ' +', ' '));

CREATE OR REPLACE TEMPORARY FUNCTION fac_place_key(s STRING) RETURNS STRING
RETURN replace(fac_place(s), ' ', '');

CREATE OR REPLACE TEMPORARY FUNCTION fac_city_hint(s STRING) RETURNS BOOLEAN
RETURN fac_norm(s) RLIKE '(^CITY OF )|( CITY$)|( CITY )';

CREATE OR REPLACE TEMPORARY FUNCTION fac_num(s STRING) RETURNS DECIMAL(20,4)
RETURN try_cast(regexp_replace(trim(s), '[ ,]', '') AS DECIMAL(20,4));

CREATE OR REPLACE TEMPORARY FUNCTION fac_parse_error(s STRING) RETURNS INT
RETURN CASE WHEN s IS NULL OR trim(s) = '' THEN 0
            WHEN try_cast(regexp_replace(trim(s), '[ ,]', '') AS DECIMAL(20,4)) IS NOT NULL THEN 0 ELSE 1 END;

CREATE OR REPLACE TEMPORARY FUNCTION fac_region_psgc(s STRING) RETURNS STRING
RETURN CASE
  WHEN fac_norm(s) RLIKE '(^| )(NCR|N C R|NATIONAL CAPITAL|METRO MANILA)( |$)'           THEN '1300000000'
  WHEN fac_norm(s) RLIKE '(^| )(CAR|C A R|CORDILLERA)( |$)'                               THEN '1400000000'
  WHEN fac_norm(s) RLIKE '(^| )(BARMM|ARMM|BANGSAMORO|AUTONOMOUS REGION IN MUSLIM MINDANAO)( |$)' THEN '1900000000'
  WHEN fac_norm(s) RLIKE '(^| )(NIR|NEGROS ISLAND)( |$)'                                  THEN '1800000000'
  WHEN fac_norm(s) RLIKE '(^| )(MIMAROPA|IV B|4 B|4B|IVB)( |$)'                           THEN '1700000000'
  WHEN fac_norm(s) RLIKE '(^| )(CARAGA|XIII|13)( |$)'                                     THEN '1600000000'
  WHEN fac_norm(s) RLIKE '(^| )(CALABARZON|IV A|4 A|4A|IVA)( |$)'                         THEN '0400000000'
  ELSE CASE regexp_extract(fac_norm(s), '^REGION ([IVX]+|[0-9]+)( |$)', 1)
         WHEN 'I' THEN '0100000000' WHEN '1' THEN '0100000000' WHEN 'II' THEN '0200000000' WHEN '2' THEN '0200000000'
         WHEN 'III' THEN '0300000000' WHEN '3' THEN '0300000000' WHEN 'V' THEN '0500000000' WHEN '5' THEN '0500000000'
         WHEN 'VI' THEN '0600000000' WHEN '6' THEN '0600000000' WHEN 'VII' THEN '0700000000' WHEN '7' THEN '0700000000'
         WHEN 'VIII' THEN '0800000000' WHEN '8' THEN '0800000000' WHEN 'IX' THEN '0900000000' WHEN '9' THEN '0900000000'
         WHEN 'X' THEN '1000000000' WHEN '10' THEN '1000000000' WHEN 'XI' THEN '1100000000' WHEN '11' THEN '1100000000'
         WHEN 'XII' THEN '1200000000' WHEN '12' THEN '1200000000' END
  END;

CREATE OR REPLACE TEMP VIEW fac_geo AS
SELECT psgc_code,
       CASE geographic_level WHEN 'Reg' THEN 'region' WHEN 'Prov' THEN 'province' WHEN 'City' THEN 'city'
            WHEN 'Mun' THEN 'municipality' WHEN 'SubMun' THEN 'submunicipality' WHEN 'Bgy' THEN 'barangay' END AS geo_level,
       city_class, fac_place_key(area_name) AS place_norm,
       concat(substr(psgc_code, 1, 2), '00000000') AS region_psgc,
       CASE WHEN geographic_level IN ('City', 'Mun') AND coalesce(city_class, '') <> 'HUC'
            THEN concat(substr(psgc_code, 1, 5), '00000') END AS province_psgc
FROM public_health.silver.geography_silver
WHERE content_checksum = geography_checksum;


-- =============================================================================
-- [1] classify: every row with a region label is a hospital record
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__classified AS
SELECT 'public_health.bronze.facilities_bed_occupancy_bronze' AS input_table, t.*,
       try_cast(trim(_source_sheet) AS INT) AS sheet_year,
       CASE WHEN Region IS NOT NULL THEN 'data' ELSE 'non_data' END AS row_class,
       CASE WHEN Region IS NULL THEN 'blank_or_note' END AS reason
FROM public_health.bronze.facilities_bed_occupancy_bronze t
WHERE _source_key = 'bed_occupancy';


-- =============================================================================
-- [2] inputs
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_bor_table_versions AS
SELECT 'public_health.bronze.facilities_bed_occupancy_bronze' AS table_name,
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.bronze.facilities_bed_occupancy_bronze)) AS table_version
UNION ALL SELECT 'public_health.silver.geography_silver',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.geography_silver))
UNION ALL SELECT 'public_health.silver.facilities_area_exceptions',
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_area_exceptions));

CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__inputs AS
WITH files AS (
  SELECT input_table, _source_key, _source_file, _source_sheet, _ingested_at, count(*) AS input_rows
  FROM public_health.silver.facilities_bed_occupancy_silver__classified
  GROUP BY input_table, _source_key, _source_file, _source_sheet, _ingested_at
)
SELECT bor_build_id AS build_id, 'public_health.silver.facilities_bed_occupancy_silver' AS target_table,
       'bronze_source' AS input_kind, f.input_table, max(v.table_version) AS input_table_version,
       f._source_key AS source_key, f._source_file AS source_file, f._source_sheet AS source_sheet,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.batch_id) END AS load_batch_id,
       CASE WHEN count(m.batch_id) = 1 THEN max(m.content_checksum) END AS content_checksum,
       f._ingested_at AS ingested_at, f.input_rows, CAST(NULL AS STRING) AS reference_version,
       count(m.batch_id) AS metadata_matches
FROM files f
LEFT JOIN public_health.bronze.ingestion_metadata m
  ON m.source_name = f._source_key AND lower(m.status) = 'processed'
 AND endswith(m.raw_file_path, concat('/', f._source_file))
 AND m.source_sheet <=> f._source_sheet AND m.retrieved_at = f._ingested_at
LEFT JOIN fac_bor_table_versions v ON v.table_name = f.input_table
GROUP BY f.input_table, f._source_key, f._source_file, f._source_sheet, f._ingested_at, f.input_rows
UNION ALL
SELECT bor_build_id, 'public_health.silver.facilities_bed_occupancy_silver', 'table', t.table_name, v.table_version,
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CASE WHEN t.table_name LIKE '%geography_silver' THEN geography_checksum END,
       CAST(NULL AS TIMESTAMP), t.n, CAST(NULL AS STRING), CAST(NULL AS BIGINT)
FROM (SELECT 'public_health.silver.geography_silver' AS table_name, count(*) AS n FROM fac_geo
      UNION ALL SELECT 'public_health.silver.facilities_area_exceptions', count(*) FROM public_health.silver.facilities_area_exceptions) t
LEFT JOIN fac_bor_table_versions v ON v.table_name = t.table_name;


-- =============================================================================
-- [3] areas
--   province unit (every year): the Province/District label -> province, or a city / NCR district
--     printed in that column (exceptions, source_group 'bor', match_type 'province_label').
--     The unit is AS PRINTED: a highly urbanized city sits under its geographic province, so a
--     2025 province unit includes those cities (geo_coverage_basis).
--   city / municipality (2020-2024 only): name within the province, else the region, else the country.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__geo AS
WITH r AS (
  SELECT _source_file, _source_sheet, _source_row_num, sheet_year,
         fac_region_psgc(Region) AS region_psgc,
         fac_norm(`Province/District`) AS prov_norm, fac_place_key(`Province/District`) AS prov_place,
         `City/Municipality` AS city_label, fac_place_key(`City/Municipality`) AS city_place,
         fac_city_hint(`City/Municipality`) AS city_hint
  FROM public_health.silver.facilities_bed_occupancy_silver__classified WHERE row_class = 'data'
),
pe AS (SELECT * FROM public_health.silver.facilities_area_exceptions WHERE source_group = 'bor' AND match_type = 'province_label'),
p_reg AS (
  SELECT r._source_sheet, r._source_row_num, count(*) AS n, max(g.psgc_code) AS psgc_code
  FROM r JOIN fac_geo g ON g.geo_level = 'province' AND g.region_psgc = r.region_psgc AND g.place_norm = r.prov_place
  GROUP BY r._source_sheet, r._source_row_num
),
p_any AS (   -- e.g. NEGROS OCCIDENTAL printed under Region VI, now in the Negros Island Region
  SELECT r._source_sheet, r._source_row_num, count(*) AS n, max(g.psgc_code) AS psgc_code
  FROM r JOIN fac_geo g ON g.geo_level = 'province' AND g.place_norm = r.prov_place
  GROUP BY r._source_sheet, r._source_row_num
),
rp AS (
  SELECT r.*,
         coalesce(pe.psgc_code, CASE WHEN pr.n = 1 THEN pr.psgc_code END, CASE WHEN pr.n IS NULL AND pa.n = 1 THEN pa.psgc_code END) AS prov_psgc,
         pe.area_key AS prov_named_key,
         CASE WHEN pe.match_value IS NOT NULL THEN 'exception' WHEN pr.n = 1 THEN 'name_level_parent'
              WHEN pr.n IS NULL AND pa.n = 1 THEN 'name_level_other_region' END AS prov_method
  FROM r
  LEFT JOIN pe ON pe.match_value = r.prov_norm
  LEFT JOIN p_reg pr ON pr._source_sheet = r._source_sheet AND pr._source_row_num = r._source_row_num
  LEFT JOIN p_any pa ON pa._source_sheet = r._source_sheet AND pa._source_row_num = r._source_row_num
),
cand AS (
  SELECT rp._source_sheet, rp._source_row_num, g.psgc_code,
         CASE WHEN g.province_psgc = rp.prov_psgc OR g.psgc_code = rp.prov_psgc THEN 4
              WHEN g.region_psgc = rp.region_psgc THEN 2 ELSE 0 END
       + CASE WHEN rp.city_hint AND g.geo_level = 'city' THEN 1
              WHEN NOT rp.city_hint AND g.geo_level IN ('municipality', 'submunicipality') THEN 1 ELSE 0 END AS score
  FROM rp JOIN fac_geo g
    ON rp.city_label IS NOT NULL AND g.geo_level IN ('city', 'municipality', 'submunicipality') AND g.place_norm = rp.city_place
),
best AS (
  SELECT _source_sheet, _source_row_num, count_if(score = max_s) AS n_best, max_by(psgc_code, score) AS best_psgc,
         max(max_s) AS best_score, array_join(array_sort(collect_list(psgc_code)), ',') AS candidates
  FROM (SELECT *, max(score) OVER (PARTITION BY _source_sheet, _source_row_num) AS max_s FROM cand)
  GROUP BY _source_sheet, _source_row_num
),
ce AS (SELECT * FROM public_health.silver.facilities_area_exceptions WHERE source_group = 'bor' AND match_type = 'area_label')
SELECT rp._source_file, rp._source_sheet, rp._source_row_num,
       coalesce(concat('psgc:', rp.prov_psgc), rp.prov_named_key) AS province_unit_key,
       rp.prov_method,
       coalesce(ce.psgc_code, CASE WHEN b.n_best = 1 THEN b.best_psgc END) AS city_psgc,
       CASE WHEN ce.psgc_code IS NOT NULL THEN 'exception'
            WHEN b.n_best = 1 AND b.best_score >= 4 THEN 'name_level_parent'
            WHEN b.n_best = 1 THEN 'name_level_other_region' END AS city_method,
       b.candidates AS city_candidates,
       rp.city_label
FROM rp
LEFT JOIN best b ON b._source_sheet = rp._source_sheet AND b._source_row_num = rp._source_row_num
LEFT JOIN ce ON ce.match_value = fac_norm(rp.city_label) AND ce.parent_label_norm <=> rp.prov_norm;


-- =============================================================================
-- [4] candidate rows
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__candidate AS
WITH d AS (
  SELECT c.*, g.province_unit_key, g.prov_method, g.city_psgc, g.city_method, g.city_candidates,
         fac_num(c.Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds) AS bor,
         -- identical rows in one sheet: the first is kept for the baseline, the others are flagged
         row_number() OVER (PARTITION BY c._source_file, c._source_sheet, c.Region, c.`Province/District`,
                                         c.`City/Municipality`, c.General, c.Nature_of_Ownership,
                                         c.Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds,
                                         c.Authorized_Bed_Capacity, c.Implementing_Beds
                            ORDER BY c._source_row_num) AS copy_no
  FROM public_health.silver.facilities_bed_occupancy_silver__classified c
  JOIN public_health.silver.facilities_bed_occupancy_silver__geo g
    ON g._source_file = c._source_file AND g._source_sheet <=> c._source_sheet AND g._source_row_num = c._source_row_num
  WHERE c.row_class = 'data'
)
SELECT
  sheet_year                                                        AS year,
  make_date(sheet_year, 1, 1)                                       AS reporting_period_start,   -- assumed
  make_date(sheet_year, 12, 31)                                     AS reporting_period_end,     -- assumed
  false                                                             AS period_confirmed,
  -- finest area available: city / municipality (2020-2024), else the province unit
  coalesce(concat('psgc:', city_psgc), province_unit_key)           AS area_key,
  coalesce(city_psgc, CASE WHEN province_unit_key LIKE 'psgc:%' THEN substr(province_unit_key, 6) END) AS psgc_code,
  CASE WHEN city_psgc IS NOT NULL THEN 'city_municipality' ELSE 'province_or_district' END AS area_level_published,
  coalesce(trim(`City/Municipality`), trim(`Province/District`))    AS geo_name_source,
  coalesce(city_method, prov_method)                                AS geo_resolution_method,
  province_unit_key,
  trim(`Province/District`)                                         AS province_label_source,
  trim(Region)                                                      AS region_as_published,
  CASE WHEN province_unit_key LIKE 'ncr-district:%' THEN 'ncr_district'
       WHEN sheet_year = 2025 THEN 'province_incl_cities_as_reported' ELSE 'city_municipality' END AS geo_coverage_basis,
  regexp_extract(General, '(Level [1-3])', 1)                       AS service_level,
  lower(trim(Nature_of_Ownership))                                  AS ownership,
  CAST(bor AS DECIMAL(7,2))                                         AS bor_pct,
  'authorized'                                                      AS bor_bed_basis,            -- per the column header
  false                                                             AS bor_basis_confirmed,
  CAST(fac_num(Authorized_Bed_Capacity) AS INT)                     AS authorized_beds,           -- 2025 only
  CAST(fac_num(Implementing_Beds) AS INT)                           AS implementing_beds,         -- 2025 only
  (bor > 100)                                                       AS is_bor_over_100,
  (copy_no > 1)                                                     AS is_exact_duplicate_in_sheet,
  CASE WHEN bor IS NULL THEN 'bor_null'
       WHEN bor < 0 THEN 'bor_negative'
       WHEN bor > bor_baseline_max THEN 'bor_gt_150'
       WHEN regexp_extract(General, '(Level [1-3])', 1) = '' OR General IS NULL THEN 'level_missing'
       WHEN copy_no > 1 THEN 'exact_duplicate'
       WHEN sheet_year < 2025 THEN 'private_only_year' END         AS exclusion_reason,
  'restricted'                                                      AS sharing,
  bor_build_id                                                      AS _silver_build_id,
  _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date,
  map_filter(map('bor_pct', 'Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds',
                 'authorized_beds', CASE WHEN sheet_year = 2025 THEN 'Authorized_Bed_Capacity' END,
                 'implementing_beds', CASE WHEN sheet_year = 2025 THEN 'Implementing_Beds' END,
                 'service_level', 'General', 'ownership', 'Nature_of_Ownership'),
             (k, v) -> v IS NOT NULL)                               AS _metric_source_columns,
  -- validation only
  fac_parse_error(Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds) + fac_parse_error(Authorized_Bed_Capacity)
    + fac_parse_error(Implementing_Beds)                            AS parse_errors,
  city_candidates,
  `City/Municipality` AS city_label_raw
FROM d;

-- baseline_eligible is derived from exclusion_reason so the two can never disagree
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__candidate2 AS
SELECT c.* EXCEPT (exclusion_reason, sharing, _silver_build_id, _source_key, _source_file, _source_sheet,
                   _source_row_num, _as_of_date, _metric_source_columns, parse_errors, city_candidates, city_label_raw),
       (exclusion_reason IS NULL) AS baseline_eligible, exclusion_reason, sharing,
       _silver_build_id, _source_key, _source_file, _source_sheet, _source_row_num, _as_of_date, _metric_source_columns,
       parse_errors, city_candidates, city_label_raw
FROM public_health.silver.facilities_bed_occupancy_silver__candidate c;


-- =============================================================================
-- [4b] dispositions (lineage-qc-contract.md v1.3, section 7.1): one row per bronze row that was
--      NOT loaded. Loaded rows are not stored: the silver rows carry their bronze row reference, and
--      rows loaded per input = input_rows - rows here (quality.v_build_reconciliation).
--      Every data row is loaded (ineligible and identical rows too: eligibility is a flag).
--      Saved before the gates, so the gates check exactly what step [9] writes.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__dispositions AS
SELECT bor_build_id AS build_id, 'public_health.silver.facilities_bed_occupancy_silver' AS target_table,
       c._source_key AS source_key, c._source_file AS source_file, c._source_sheet AS source_sheet,
       c._source_row_num AS source_row_num, i.load_batch_id, i.content_checksum,
       'dropped_non_data' AS disposition, c.reason, CAST(NULL AS STRING) AS target_key,
       current_timestamp() AS recorded_at
FROM public_health.silver.facilities_bed_occupancy_silver__classified c
LEFT JOIN public_health.silver.facilities_bed_occupancy_silver__inputs i
  ON i.input_kind = 'bronze_source' AND i.source_key = c._source_key AND i.source_file = c._source_file
 AND i.source_sheet <=> c._source_sheet
WHERE c.row_class <> 'data';


-- =============================================================================
-- [5] gates
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver__gates AS
SELECT 'inputs: the six sheets 2020-2025 are read, once each' AS gate, 'fail' AS severity,
       CASE WHEN (SELECT count(*) FROM public_health.silver.facilities_bed_occupancy_silver__inputs
                  WHERE input_kind = 'bronze_source' AND source_sheet IN ('2020','2021','2022','2023','2024','2025')) = 6
             AND (SELECT count(*) FROM public_health.silver.facilities_bed_occupancy_silver__inputs WHERE input_kind = 'bronze_source') = 6
            THEN 0 ELSE 1 END AS failures
UNION ALL
SELECT 'inputs: more than one ingestion log match', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__inputs WHERE input_kind = 'bronze_source' AND metadata_matches > 1
UNION ALL
SELECT 'inputs: no processed ingestion log match (strict_load_lookup = true)', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__inputs
  WHERE input_kind = 'bronze_source' AND metadata_matches = 0 AND strict_load_lookup
UNION ALL
SELECT 'sheet name is not a year', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__classified WHERE sheet_year IS NULL
UNION ALL
SELECT 'region label not recognised', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__classified
  WHERE row_class = 'data' AND fac_region_psgc(Region) IS NULL
UNION ALL
SELECT 'province / district not resolved (add a bor province_label exception)', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE province_unit_key IS NULL
UNION ALL
SELECT 'cell that is not a number', 'fail', coalesce(sum(parse_errors), 0)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2
UNION ALL
SELECT 'ownership not government / private', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE ownership NOT IN ('government', 'private') OR ownership IS NULL
UNION ALL
SELECT 'key not unique', 'fail', count(*) FROM (
  SELECT 1 FROM public_health.silver.facilities_bed_occupancy_silver__candidate2
  GROUP BY _source_key, _source_file, _source_sheet, _source_row_num HAVING count(*) > 1)
UNION ALL
SELECT 'lineage: a lineage column is null', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2
  WHERE _silver_build_id IS NULL OR _source_key IS NULL OR _source_file IS NULL OR _source_sheet IS NULL
     OR _source_row_num IS NULL OR _as_of_date IS NULL OR _metric_source_columns IS NULL
UNION ALL
-- contract 7.1, per input: distinct bronze rows referenced by silver = rows read - rows not loaded
SELECT 'reconciliation: inputs where rows referenced by silver <> rows read - rows not loaded', 'fail', count(*)
FROM public_health.silver.facilities_bed_occupancy_silver__inputs i
LEFT JOIN (
  SELECT _source_key, _source_file, _source_sheet, count(*) AS n FROM (
    SELECT DISTINCT _source_key, _source_file, _source_sheet, _source_row_num FROM public_health.silver.facilities_bed_occupancy_silver__candidate2)
  GROUP BY _source_key, _source_file, _source_sheet) ref
  ON ref._source_key = i.source_key AND ref._source_file = i.source_file AND ref._source_sheet <=> i.source_sheet
LEFT JOIN (
  SELECT source_key, source_file, source_sheet, count(*) AS n
  FROM public_health.silver.facilities_bed_occupancy_silver__dispositions GROUP BY source_key, source_file, source_sheet) nl
  ON nl.source_key = i.source_key AND nl.source_file = i.source_file AND nl.source_sheet <=> i.source_sheet
WHERE i.input_kind = 'bronze_source' AND i.input_rows - coalesce(nl.n, 0) <> coalesce(ref.n, 0)
UNION ALL
SELECT 'output: silver rows <> bronze data rows', 'fail',
       abs((SELECT count(*) FROM public_health.silver.facilities_bed_occupancy_silver__candidate2)
         - (SELECT count_if(row_class = 'data') FROM public_health.silver.facilities_bed_occupancy_silver__classified))
UNION ALL
SELECT 'control: 2025 rows = 1,162 (418 government, 744 private)', 'fail',
       CASE WHEN (SELECT count_if(ownership = 'government') FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE year = 2025) = 418
             AND (SELECT count_if(ownership = 'private') FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE year = 2025) = 744
            THEN 0 ELSE 1 END
UNION ALL
SELECT 'control: 2025 authorized beds = 109,251', 'fail',
       CASE WHEN (SELECT sum(authorized_beds) FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE year = 2025) = 109251
            THEN 0 ELSE 1 END
UNION ALL
SELECT 'control: 2020-2024 sheets are private only', 'fail', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE year < 2025 AND ownership <> 'private'
UNION ALL
SELECT 'warn: 2020-2024 city / municipality not resolved (record kept at province level)', 'warn', count(*)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2 WHERE year < 2025 AND area_level_published <> 'city_municipality'
UNION ALL
SELECT 'warn: BOR above 100 (kept, flagged)', 'warn', count_if(is_bor_over_100)
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2
UNION ALL
SELECT 'warn: BOR above the baseline maximum (kept, not baseline-eligible)', 'warn', count_if(exclusion_reason = 'bor_gt_150')
  FROM public_health.silver.facilities_bed_occupancy_silver__candidate2;


-- =============================================================================
-- [6] build log row   [7] stop if a gate failed
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_bor_build_log AS
SELECT bor_build_id AS build_id, 'silver' AS layer, 'facilities' AS domain,
       'public_health.silver.facilities_bed_occupancy_silver' AS target_table,
       silver_build_started_at AS started_at, current_timestamp() AS finished_at,
       CASE WHEN g.failed_gates = 0 THEN 'succeeded' ELSE 'failed' END AS status,
       CASE WHEN g.failed_gates = 0 THEN (SELECT count(*) FROM public_health.silver.facilities_bed_occupancy_silver__candidate2) END AS row_count,
       CAST(NULL AS BIGINT) AS target_table_version, code_version AS code_version,
       to_json(named_struct('strict_load_lookup', strict_load_lookup, 'bor_baseline_max', bor_baseline_max,
                            'geography_checksum', geography_checksum, 'warnings', g.warn_names)) AS parameters,
       job_run_id AS run_id,
       CASE WHEN g.failed_gates > 0 THEN substr(concat('gates failed: ', g.failed_names), 1, 500) END AS error_message
FROM (SELECT count_if(failures <> 0 AND severity = 'fail') AS failed_gates,
             concat_ws(' | ', collect_list(CASE WHEN failures <> 0 AND severity = 'fail' THEN concat(gate, ' = ', failures) END)) AS failed_names,
             concat_ws(' | ', collect_list(CASE WHEN failures <> 0 AND severity = 'warn' THEN concat(gate, ' = ', failures) END)) AS warn_names
      FROM public_health.silver.facilities_bed_occupancy_silver__gates) g;

INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
       target_table_version, code_version, parameters, run_id, error_message
FROM fac_bor_build_log WHERE status = 'failed';

SELECT raise_error(concat('facilities_bed_occupancy_silver was NOT published. ', error_message))
FROM fac_bor_build_log WHERE status = 'failed';


-- =============================================================================
-- [8] write (full rebuild). Keep access to this table restricted.
-- =============================================================================
CREATE OR REPLACE TABLE public_health.silver.facilities_bed_occupancy_silver AS
SELECT * EXCEPT (parse_errors, city_candidates, city_label_raw)
FROM public_health.silver.facilities_bed_occupancy_silver__candidate2;


-- =============================================================================
-- [9] record the build
-- =============================================================================
INSERT INTO public_health.quality.build_log
  (build_id, layer, domain, target_table, started_at, finished_at, status, row_count,
   target_table_version, code_version, parameters, run_id, error_message)
SELECT build_id, layer, domain, target_table, started_at, finished_at, status,
       (SELECT count(*) FROM public_health.silver.facilities_bed_occupancy_silver),
       (SELECT max(version) FROM (DESCRIBE HISTORY public_health.silver.facilities_bed_occupancy_silver)),
       code_version, parameters, run_id, error_message
FROM fac_bor_build_log;

INSERT INTO public_health.quality.build_inputs
  (build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
   source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version)
SELECT build_id, target_table, input_kind, input_table, input_table_version, source_key, source_file,
       source_sheet, load_batch_id, content_checksum, ingested_at, input_rows, reference_version
FROM public_health.silver.facilities_bed_occupancy_silver__inputs;

-- bronze rows not loaded (staged in [4b]); loaded rows are not written (contract 7.1)
INSERT INTO public_health.quality.row_disposition
  (build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
   content_checksum, disposition, reason, target_key, recorded_at)
SELECT build_id, target_table, source_key, source_file, source_sheet, source_row_num, load_batch_id,
       content_checksum, disposition, reason, target_key, recorded_at
FROM public_health.silver.facilities_bed_occupancy_silver__dispositions;


-- =============================================================================
-- [10] summary: the candidate baseline by year (aggregates only; the rows are restricted)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW fac_bor_summary AS
SELECT year, ownership, count(*) AS records, count_if(baseline_eligible) AS baseline_eligible,
       count_if(is_bor_over_100) AS bor_over_100, sum(authorized_beds) AS authorized_beds,
       -- bed-weighted occupancy of eligible 2025 records (same period for every record assumed)
       round(sum(CASE WHEN baseline_eligible THEN bor_pct * authorized_beds END)
             / sum(CASE WHEN baseline_eligible THEN authorized_beds END), 1) AS weighted_bor_eligible
FROM public_health.silver.facilities_bed_occupancy_silver
GROUP BY year, ownership;

CREATE OR REPLACE TEMP VIEW fac_bor_warnings AS
SELECT gate, failures FROM public_health.silver.facilities_bed_occupancy_silver__gates WHERE severity = 'warn';

SELECT * FROM fac_bor_warnings;

DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__candidate;
DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__candidate2;
DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__gates;
DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__inputs;
DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__geo;
DROP TABLE IF EXISTS public_health.silver.facilities_bed_occupancy_silver__dispositions;

SELECT * FROM fac_bor_summary ORDER BY year, ownership;